In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Data: selection functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/data_selection.ipynb)

Every number on this page is written by `scripts/paper_text_data.py` from the result tables and catalogues
(`results/macros_data.tex`); the code cells are the figure scripts `scripts/paper_selection.py`, `scripts/paper_selection_checks.py`
and `scripts/paper_selection_gaps.py`, step by step, and the scripts that compute the weights and their tests, shown but not run.

The stellar mass of the next chapters counts giants per metallicity bin, so we need the probability that an LMC giant entered each
spectroscopic sample, and in particular whether that probability depends on metallicity. This probability, as a function of a
star's properties, is the selection function of the sample. For the RR Lyrae we measure the completeness of the union of two
independent surveys. For the giants we reproduce the targeting of each survey and weight its spectra in four steps:

1. **Parent.** We re-apply the survey's own targeting rules to the catalogue it targeted from (*Gaia* DR3 for BOSS, the Two Micron
   All Sky Survey, 2MASS, for APOGEE). The parent holds every star the survey could have observed (Table 2.1).
2. **Selection groups.** We divide the parent into groups of stars the survey treated alike (sky position, magnitude, targeting
   class) and weight each good spectrum by the number of parent stars per good spectrum in its group.
3. **Pass factor.** Spectra of metal-poor giants fail more often. A metallicity known for observed and unobserved stars alike, the
   *Gaia* XP estimate (from the low-resolution blue and red prism spectra of *Gaia*), measures this, and a pass factor puts the
   failed spectra back.
4. **Coverage.** Groups with fewer than 3 good spectra are uncovered; a coverage factor puts their parent
   stars back.

Non-members (dwarfs and stars outside the LMC velocity range) stay in the sample until the weights are made and are removed
afterwards. The Milky Way giants that share the LMC velocities are subtracted statistically in the stellar-mass chapter. The weights
rest on five assumptions, each tested below:

- **A1** the parent contains every LMC giant inside the survey's selection;
- **A2** within a selection group, the survey chose which stars to observe without regard to their metallicity or membership;
- **A3** whether an observed star gives a good spectrum depends on its metallicity only through its magnitude, XP metallicity and
  ring;
- **A4** the parent stars of uncovered groups have the metallicities and member fraction of the covered stars in the same ring and
  magnitude bin;
- **A5** non-members pass the spectroscopic quality cuts at the same rate as members of the same pass group.

| | BOSS (carton mwm_magcloud_rgb_boss) | APOGEE-2 (LMC1–35) |
|---|---|---|
| Source catalogue | *Gaia* DR3 | 2MASS |
| Magnitude | $G$ ≤ 17.5 | 12.35 ≤ $H$ ≤ $H_{\rm lim}$ (14.5–15.3) |
| Colour | $G_{\rm BP}-G_{\rm RP}$ ≥ 1 | $J-K_s$ between the polygon edges at $H$ (observed) |
| Colour–magnitude | dereddened $(G,\,G_{\rm BP}-G_{\rm RP})$ inside the carton polygon | – |
| Proper motion | $\lvert\mu_{\alpha*}\rvert,\lvert\mu_\delta\rvert$ ≤ 3 mas/yr; within 1.2 mas/yr of (1.8, 0.40) mas/yr (Magellanic Stream coordinates) | as BOSS (a) |
| Parallax | not ($\varpi$ > 0 and $(\varpi+$ 0.025$)/\sigma_\varpi$ > 5) | as BOSS (a) |
| Sky | $\rho$ < 12° | within 0.95° of a field centre |
| Other | finite photometry and astrometry | no neighbour ≥ 2 mag brighter in $H$ within 6″ |
| Good spectrum | BOSSNet flag unset, finite [Fe/H] and $v$, S/N ≥ 16, $\sigma_v$ < 10 km/s | STAR_BAD unset, finite [Fe/H], S/N ≥ 30 |

In [ ]:
import json
import pandas as pd
# Counts at each selection step: BOSS (b) and APOGEE-2 (a) selection summaries, members of the mass sample (m)
b = json.load(open('results/boss_selection.json'))['summary']
a = json.load(open('results/apogee_selection.json'))['summary']
m = json.load(open('results/giant_counts.json'))
# Parent stars with a BOSS spectrum
b_obs = int(pd.read_parquet('data/boss_weights.parquet', columns=['observed']).observed.sum())
# One row per step, one column per survey
pd.DataFrame([('Parent stars', b['eligible'], a['candidates']), ('Observed', b_obs, a['observed']),
              ('Good spectra', b['good'], a['quality']), ('Parent in covered groups', b['covered'], a['covered']),
              ('Mass sample (b)', m['boss_members'], m['apogee_members'])],
             columns=['', 'BOSS', 'APOGEE-2']).set_index('')

**Table 2.1.** Parent catalogues and spectroscopic samples of the giant selection functions. Upper part: the cuts, those of the
surveys' own targeting re-applied to their source catalogues except where marked; $G$, $G_{\rm BP}$, $G_{\rm RP}$: *Gaia*
magnitudes; $J$, $H$, $K_s$: 2MASS magnitudes; $H_{\rm lim}$: faint limit of the field; $\mu_{\alpha*}$, $\mu_\delta$: proper
motions; $\varpi$, $\sigma_\varpi$: parallax and its error (mas); $\sigma_v$: velocity error. (a) Not an APOGEE cut: in LMC1–17 a
stand-in for the unpublished Washington+DDO51 giant flag, in LMC18–35 a foreground filter. Lower part: numbers of stars within
12° (BOSS) and 0.95° of the 35 field centres (APOGEE), from
`results/boss_selection.json`, `results/apogee_selection.json` and `results/giant_counts.json` (as `scripts/paper_table_cuts.py`).
(b) Giants ($T_{\rm eff}$ < 5500 K, $\log g$ < 3.5) with 150 <
$v_{\rm los}$ < 420 km/s and a positive weight.

<details><summary>About <code>scripts/paper_selection.py</code></summary>

```text
Paper figure (Sect. 2.2): the selection functions of the three samples.

(a) RRab completeness against distance from the LMC centre: Gaia = fraction of the RRab found by OGLE's own search that Gaia DR3
also classifies as RRab; OGLE = fraction of the Gaia RRab inside the OGLE-IV footprint found by OGLE's own search (stars from
OGLE-LMC-RRLYR-39960 on were added from Gaia); union = n_union / N, with N the Lincoln-Petersen estimate n_O n_G / n_both made in
magnitude quartiles (results/rrab_independence.csv, scripts/check_rrab_independence.py). Binomial errors; bins with >= 20 stars.
(b, c) Share of the spectroscopic sample in each bin of Gaia XP [M/H] (Andrae et al. 2023, cool giants; measured for observed and
unobserved stars alike) divided by the share of the parent: good spectra unweighted (open) and with the final weights (filled).
(b) BOSS: parent = eligible stars of mwm_magcloud_rgb_boss in covered selection groups; w = selection-group weight x pass factor
(scripts/boss_selection.py). (c) APOGEE: parent = covered candidates of the LMC1-35 RGB targeting (scripts/apogee_selection.py).
Errors: bootstrap over nside-16 pixels (weights fixed); bins with >= NMIN sample stars.
Writes figures/paper_selection.png.

Inputs: data/rrab.parquet (release_rrab; flags ogle_own, in_gaia, rho [deg]), results/rrab_independence.csv
(scripts/check_rrab_independence.py; per ring rho_lo, rho_hi, C_union_by_mag), data/boss_weights.parquet (release_boss_weights;
BOSS parent with group, good, w, mh_xp), data/apogee_weights.parquet (release_apogee_weights; APOGEE candidates with covered,
quality, w, mh_xp).
Panels (1 x 3, square; legends above the panels):
(a) x: rho [deg], 0-16; y: completeness, 0-1.04; Gaia (blue) and OGLE own search (orange) in 1-deg bins with >= 20 stars,
    binomial errors; green squares: union (capture-recapture), x error bars the ring half-widths.
(b) x: XP [M/H], -2.6 to 0.1; y: sample / parent share per bin for BOSS (good spectra), unweighted (open) and with the final
    weights w (filled, offset +0.03); errors: standard deviation of NBOOT nside-16 pixel bootstraps; bins with >= NMIN sample
    stars; grey line at 1.
(c) the same for APOGEE (good-quality spectra among the covered candidates).
Placement: Sect. 2.2, main text (docs/figures.md). Run: .venv/bin/python scripts/paper_selection.py   (about 4 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_selection.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/raw/ogle4/fields-OGLE-IV-LMC.txt` | public archive | OGLE-IV field centres, LMC (LMC); Udalski et al. 2015, AcA 65, 1 |
| `data/raw/ogle4/fields-OGLE-IV-MBR.txt` | public archive | OGLE-IV field centres, MBR (Magellanic Bridge); Udalski et al. 2015, AcA 65, 1 |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/rrab_independence.csv` | repository | completeness of the OGLE-IV and Gaia RR Lyrae catalogues and their union; written by `python scripts/check_rrab_independence.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import healpy as hp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold import ogle
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

plt.style.use(ROOT / "src" / "lmcold" / "paper.mplstyle")
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"   # colours: Gaia / BOSS, OGLE / APOGEE, union
# XP [M/H] bin edges [dex] of panels (b) BOSS and (c) APOGEE
EDGES = {"BOSS": np.array([-2.5, -2.0, -1.75, -1.5, -1.25, -1.0, -0.75, -0.5, 0.0]),
         "APOGEE": np.array([-2.5, -1.5, -1.0, -0.75, -0.5, 0.0])}  # APOGEE has ~50 metal-poor stars with XP
NMIN = 10  # minimum number of sample stars in an XP bin
NBOOT = 200  # pixel bootstrap draws
LEG_TOP = 1.36  # legends above the panels, tops aligned

## Load the data
The RRab catalogue `data/rrab.parquet` and the selection weights of the two spectroscopic samples, `data/boss_weights.parquet`
and `data/apogee_weights.parquet` (Zenodo bundle 1); the union completeness per ring from `results/rrab_independence.csv`
(written by `scripts/check_rrab_independence.py`).

In [ ]:
# data/rrab.parquet: all RRab within 20 deg, with the flags ogle_own, in_gaia and rho [deg]
t = pd.read_parquet(DATA / "rrab.parquet")
# data/boss_weights.parquet: the BOSS parent with selection group, good-spectrum flag, weight w and XP [M/H]
b = pd.read_parquet(DATA / "boss_weights.parquet")
# data/apogee_weights.parquet: the APOGEE candidates (only the covered ones kept) with quality flag, weight w and XP [M/H]
a = pd.read_parquet(DATA / "apogee_weights.parquet").query("covered")
# results/rrab_independence.csv: union completeness per ring (rho_lo, rho_hi, C_union_by_mag), from magnitude-quartile estimates
u = pd.read_csv(ROOT / "results" / "rrab_independence.csv")

## The samples
All RRab within 20 deg with the OGLE-IV footprint flag; the BOSS parent in covered selection groups.

In [ ]:
t = t[t.rho < 20].copy()  # the whole RRab catalogue (Gaia query radius 20 deg), not only members
# True for RRab inside the OGLE-IV footprint (lmcold.ogle)
t["in_fp"] = ogle.in_footprint(t.ra.to_numpy(), t.dec.to_numpy())
b = b[b.group.map(b.groupby("group").good.sum()) >= 3]  # covered selection groups (>= 3 good spectra, selection.NMIN_GOOD)

## Completeness and share ratios
`binned` gives a completeness per bin of $\rho$; `ratios` gives the share of the sample in each XP [M/H] bin over
the share of the parent, with a bootstrap over HEALPix pixels.

In [ ]:
def binned(s, flag, col, edges):
    """Fraction of s with `flag` in bins of `col`, binomial error; NaN where fewer than 20 stars.

    Parameters
    ----------
    s : pandas.DataFrame
        Stars.
    flag : str
        Boolean column whose mean is the fraction.
    col : str
        Binned column (here rho [deg]).
    edges : numpy.ndarray
        Bin edges in `col`.

    Returns
    -------
    centres, fraction, error : numpy.ndarray
        Bin centres, fraction with `flag` and sqrt(f (1 - f) / n) per bin.
    """
    # Bin index of each star (-1 or len(edges) - 1 outside); n: stars per bin
    k = np.digitize(s[col], edges) - 1
    n = np.array([(k == i).sum() for i in range(len(edges) - 1)])
    # Fraction with flag per bin (NaN below 20 stars); returned with bin centres and binomial errors
    f = np.array([s[flag][k == i].mean() if n[i] >= 20 else np.nan for i in range(len(edges) - 1)])
    return 0.5 * (edges[1:] + edges[:-1]), f, np.sqrt(f * (1 - f) / np.maximum(n, 1))


def ratios(t, inc, weights, edges):
    """Sample / parent share per XP [M/H] bin for each weight column (None = unweighted), pixel-bootstrap errors, sample count
    per bin.

    Parameters
    ----------
    t : pandas.DataFrame
        Parent stars with ra, dec [deg], mh_xp and the columns named in `inc` and `weights`; stars without XP [M/H] are dropped.
    inc : str
        Boolean column marking the sample (good spectra).
    weights : list of (str or None)
        Weight columns of the sample; None for unit weights.
    edges : numpy.ndarray
        XP [M/H] bin edges; stars outside them are ignored.

    Returns
    -------
    ratio : numpy.ndarray, shape (len(weights), len(edges) - 1)
        (weighted sample in bin / weighted sample) / (parent in bin / parent).
    error : numpy.ndarray, same shape
        Standard deviation over NBOOT draws of nside-16 (nested) pixels with replacement.
    n : numpy.ndarray
        Unweighted sample count per bin.
    """
    # Stars with XP [M/H]; k: XP bin index, ok: inside the edges
    t = t[t.mh_xp.notna()]
    k = np.digitize(t.mh_xp, edges) - 1
    ok = (k >= 0) & (k < len(edges) - 1)
    # HEALPix nside-16 nested pixel of each star (colatitude, longitude in rad), the bootstrap unit
    pix = hp.ang2pix(16, np.radians(90 - t.dec.to_numpy()), np.radians(t.ra.to_numpy()), nest=True)
    # s: sample stars in a bin; ws: one weight array per entry of weights (ones for None)
    s = t[inc].to_numpy() & ok
    ws = [np.ones(len(t)) if w is None else t[w].to_numpy() for w in weights]
    # Occupied pixels and a seeded generator (seed 1: the same errors on every run)
    upix, rng = np.unique(pix), np.random.default_rng(1)

    def one(m):
        """Ratios for star multiplicities m (pixel draw counts; ones for the data)."""
        # Parent count per bin, then for each weighting the sample share over the parent share
        par = np.bincount(k[ok], m[ok], len(edges) - 1)
        return [np.bincount(k[s], (m * x)[s], len(edges) - 1) / np.sum((m * x)[s]) / (par / par.sum()) for x in ws]
    # Each draw: NBOOT times, pixels drawn with replacement; a star's multiplicity = the times its pixel was drawn
    boot = [one(np.bincount(np.searchsorted(upix, rng.choice(upix, len(upix))), minlength=len(upix))[np.searchsorted(upix, pix)]
                .astype(float)) for _ in range(NBOOT)]
    # Ratios of the data (multiplicity 1), bootstrap standard deviation, unweighted sample count per bin
    return np.array(one(np.ones(len(t)))), np.array(boot).std(0), np.bincount(k[s], minlength=len(edges) - 1)

## The figure
(a) RR Lyrae completeness of each survey and of their union; (b, c) BOSS and APOGEE share ratios, unweighted and
with the final weights.

In [ ]:
# Three equal panels; (a) RRab completeness
fig, ax = panel_grid(1, 3, hgap=0.5, top=0.75)
p = ax[0, 0]
# Gaia: fraction of the OGLE own-search RRab that Gaia has; OGLE: fraction of the Gaia RRab in the
# OGLE-IV footprint found by OGLE's own search; 1-deg bins of rho from 0 to 16 deg
for s, flag, col, lab in ((t[t.ogle_own], "in_gaia", BLUE, "Gaia"), (t[t.in_gaia & t.in_fp], "ogle_own", ORANGE, "OGLE (own search)")):
    x, f, e = binned(s, flag, "rho", np.arange(0, 16.1, 1.0))
    p.errorbar(x, f, yerr=e, fmt="o-", ms=2.5, lw=1, color=col, label=lab)
# Union completeness per ring (capture-recapture), x bars the ring half-widths
p.errorbar(0.5 * (u.rho_lo + u.rho_hi), u.C_union_by_mag, xerr=0.5 * (u.rho_hi - u.rho_lo), fmt="s", ms=3, lw=1, color=GREEN,
           label="OGLE or Gaia")
# Axes, legend above the panel, panel letter
p.set(xlim=(0, 16), ylim=(0, 1.04), xlabel=r"$\rho$ [deg]", ylabel="completeness")
p.legend(loc="upper left", bbox_to_anchor=(0, LEG_TOP), ncol=1, handlelength=1.5, title="RRab", alignment="left")
p.text(0.95, 0.05, "(a)", transform=p.transAxes, ha="right")
# (b) BOSS and (c) APOGEE: (parent table, sample flag, label, colour, panel letter)
for p, (s, inc, lab, col, tag) in zip(ax[0, 1:], ((b, "good", "BOSS", BLUE, "(b)"), (a, "quality", "APOGEE", ORANGE, "(c)"))):
    # Share ratios unweighted (row 0) and with weights w (row 1); x: bin centres; ok: bins with >= NMIN sample stars
    r, e, n = ratios(s, inc, [None, "w"], EDGES[lab])
    x, ok = 0.5 * (EDGES[lab][1:] + EDGES[lab][:-1]), n >= NMIN
    for i, style in enumerate((dict(fmt="o", mfc="white", lw=0.8, label="unweighted"), dict(fmt="o-", lw=1, label="final weights"))):
        # Unweighted offset by -0.03, weighted by +0.03 dex
        p.errorbar(x[ok] + 0.03 * (2 * i - 1), r[i][ok], yerr=e[i][ok], ms=3, color=col, **style)
    # Grey line at ratio 1 (sample follows the parent); y range wide enough for every error bar
    p.axhline(1, color="0.6", lw=0.6)
    p.set(xlabel=r"[M/H]$_{\rm XP}$", ylabel="sample / parent", xlim=(-2.6, 0.1),
          ylim=(0.4, max(1.7, float(np.nanmax((r + e)[:, ok])) * 1.05)))
    p.legend(loc="upper left", bbox_to_anchor=(0, LEG_TOP), ncol=1, handlelength=1.5, title=lab, alignment="left")
    p.text(0.95, 0.93, tag, transform=p.transAxes, ha="right", va="top")
fig.savefig(ROOT / "figures" / "paper_selection.png")

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_selection.png')

**Figure 2.1.** Selection functions. (a) RRab completeness against angular distance $\rho$ from the centre (deg):
*Gaia* (blue; fraction of OGLE's own RRab that *Gaia* classifies as RRab), OGLE's own search (orange; fraction of *Gaia* RRab inside
the OGLE-IV footprint found by OGLE) in 1° bins with at least 20 stars, binomial errors; and their union (green squares;
Lincoln–Petersen estimate in magnitude quartiles, horizontal bars the ring widths). (b, c) Share of the spectroscopic sample in each
bin of XP metallicity [M/H]$_{\rm XP}$ ([Andrae et al. 2023](https://doi.org/10.3847/1538-4365/acd53e)), divided by the share of the
parent, for the good spectra unweighted (open circles) and with the final weights (filled circles, joined): (b) BOSS (blue), parent =
the re-implemented carton in covered selection groups; (c) APOGEE (orange), parent = the covered candidates of the LMC1–35
targeting. Grey line: ratio 1. Errors: bootstrap over HEALPix pixels at resolution parameter $N_{\rm side}$ = 16 with the weights
fixed; bins with at least 10 spectra. `scripts/paper_selection.py`

## RR Lyrae

OGLE's own search and *Gaia* found RRab independently, so their overlap measures the completeness of each. Within the OGLE-IV
footprint, the fraction of OGLE RRab that *Gaia* also classifies as RRab is the *Gaia* completeness, and the fraction of *Gaia* RRab
that OGLE found is the OGLE completeness. If the two surveys miss stars independently, the number of RRab in a region is the
Lincoln–Petersen (capture–recapture) estimate $N=n_{\rm O}n_{\rm G}/n_{\rm OG}$, from the numbers found by OGLE ($n_{\rm O}$), by
*Gaia* ($n_{\rm G}$) and by both ($n_{\rm OG}$). We make this estimate in quartiles of mean magnitude, which removes most of the bias
from stars that both surveys find hard; splitting by period or amplitude instead changes the union completeness by at most
1.3 per cent.

Figure 2.1 (a) shows the completeness against $\rho$. *Gaia* misses 39 per cent of the RRab at
the crowded centre and at most 2 per cent beyond 9°; OGLE's own search is
94–97 per cent complete inside 6° and falls beyond, where many stars
entered the OGLE collection only from *Gaia*. The union is 96 per cent complete at $\rho$ <
1°, 98.5–99.2 per cent at
1°–3° and 99.2–99.8 per cent beyond. In the
crowded centre this is an upper limit: stars that blending hides from both surveys are invisible to capture–recapture. The checks
below give the completeness against magnitude and on the sky.

### Capture–recapture estimates

`scripts/check_rrab_independence.py` makes the Lincoln–Petersen estimates in rings of $\rho$ inside the OGLE-IV footprint, once
for the whole ring and once summed over quartiles of mean magnitude, of period or of amplitude. The sums over magnitude quartiles
give the union completeness of Figure 2.1 (a); the other two give the change quoted above.

<details><summary>About <code>scripts/check_rrab_independence.py</code></summary>

```text
Check of the capture-recapture assumption: do OGLE (own search) and Gaia miss RRab independently?

Within each radial ring inside the OGLE-IV footprint, the Lincoln-Petersen total N = n_O n_G / n_both is compared with the sum
of the estimates made separately in quartiles of a stellar property. If detection probabilities vary with that property in both
surveys (heterogeneity), the unstratified estimate is biased low and the stratified sum is larger. Properties: period (measured by
both surveys), mean magnitude (Gaia G where Gaia has the star, else OGLE I + 0.6), and amplitude (OGLE I-band amplitude where
available, else Gaia peak-to-peak G scaled to I by the median ratio on the stars with both). Writes results/rrab_independence.csv.

Input: data/rrab.parquet (release_rrab); OGLE-IV footprint from ogle4_fields_lmc / ogle4_fields_mbr (lmcold.ogle.in_footprint).
OGLE detections are ogle_own (OGLE's own discoveries, independent of Gaia), Gaia detections in_gaia.
Output columns per ring (rho_lo, rho_hi [deg]): n_union (stars found by either), N_lp (unstratified Lincoln-Petersen total),
N_lp_by_<P|mag|amp> (sum over quartiles), C_union and C_union_by_<prop> = n_union / estimate (completeness of the union).
Its C_union_by_mag column is the union completeness (green squares) of figures/paper_selection.png (a); the other columns give
the largest change from splitting by period or amplitude, quoted in the selection-function text (scripts/paper_text_data.py).

Run: .venv/bin/python scripts/check_rrab_independence.py   (after build_rrab.py)
```
</details>

*Shown, not run here: data/rrab.parquet and the OGLE-IV field outlines (scripts/fetch_ogle4_fields.py); it writes results/rrab_independence.csv, read by scripts/paper_selection.py. The cells below are the script itself; run it from the repository with* `python scripts/check_rrab_independence.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import numpy as np
import pandas as pd

from lmcold import ogle
from lmcold.sources import DATA, ROOT
```

#### Load the data
The RR Lyrae catalogue `data/rrab.parquet` (`scripts/build_rrab.py`).

```python
# One row per RRab: ra, dec [deg], rho [deg from the LMC centre], P [d], I, phot_g_mean_mag [mag], amp_I, peak_to_peak_g [mag],
# ogle_own, in_gaia (detection flags).
t = pd.read_parquet(DATA / "rrab.parquet")
```

#### RRab inside the OGLE-IV footprint
Stars within 12 deg and inside the OGLE-IV fields, with a mean magnitude on the
$G$ scale and an amplitude on the $I$ scale for every star.

```python
t = t[t.rho < 12].copy()  # deg; the outer ring edge below
t = t[ogle.in_footprint(t.ra.to_numpy(), t.dec.to_numpy())]   # inside the OGLE-IV LMC and bridge fields
both = t.amp_I.notna() & t.peak_to_peak_g.notna()   # stars with both amplitudes, for the G-to-I amplitude ratio
# Mean magnitude on the G scale: Gaia G, else OGLE I + 0.6 mag (fixed offset adopted for RRab here).
t["mag"] = t.phot_g_mean_mag.astype(float).fillna(t.I + 0.6)
# Amplitude on the I scale [mag]: OGLE amp_I, else Gaia peak-to-peak G times the median amp_I / peak-to-peak G ratio.
t["amp"] = t.amp_I.fillna(t.peak_to_peak_g * np.median(t.amp_I[both] / t.peak_to_peak_g[both]))
```

#### The Lincoln-Petersen estimate
$N=n_{\rm O}n_{\rm G}/n_{\rm OG}$ from OGLE's own detections and the Gaia detections.

```python
def lp(s):
    """Lincoln-Petersen estimate n_O n_G / n_both of the total number of RRab in subsample s (NaN if n_both = 0).

    Parameters
    ----------
    s : pandas.DataFrame
        Rows with boolean columns ogle_own and in_gaia.

    Returns
    -------
    float
    """
    # Stars found by OGLE's own search, by Gaia, and by both.
    nO, nG, nB = s.ogle_own.sum(), s.in_gaia.sum(), (s.ogle_own & s.in_gaia).sum()
    # N = n_O n_G / n_both; NaN when no star is found by both (estimate undefined).
    return nO * nG / nB if nB else np.nan
```

#### Estimates per ring
In each ring, the unstratified estimate and the sums over quartiles of period, magnitude and amplitude,
and the completeness of the union; written to `results/rrab_independence.csv`.

```python
rows = []   # one dict per ring
# Rings in rho [deg].
for lo, hi in ((0, 1), (1, 2), (2, 3), (3, 6), (6, 12)):
    s = t[(t.rho >= lo) & (t.rho < hi)]
    # Stars found by either survey, and the unstratified estimate.
    row = dict(rho_lo=lo, rho_hi=hi, n_union=int((s.ogle_own | s.in_gaia).sum()), N_lp=lp(s))
    for prop in ("P", "mag", "amp"):
        q = pd.qcut(s[prop], 4, labels=False, duplicates="drop")  # quartile of the property within the ring
        # Sum of the estimates made separately in each quartile.
        row[f"N_lp_by_{prop}"] = sum(lp(s[q == k]) for k in range(int(q.max()) + 1))
    # Completeness of the union: n_union over each estimate.
    row.update({f"C_union_by_{p}": row["n_union"] / row[f"N_lp_by_{p}"] for p in ("P", "mag", "amp")}, C_union=row["n_union"] / row["N_lp"])
    rows.append(row)
r = pd.DataFrame(rows)   # one row per ring
r.to_csv(ROOT / "results" / "rrab_independence.csv", index=False, float_format="%.4f")
print(r[["rho_lo", "rho_hi", "n_union", "N_lp", "N_lp_by_P", "N_lp_by_mag", "N_lp_by_amp", "C_union", "C_union_by_P",
         "C_union_by_mag", "C_union_by_amp"]].to_string(index=False, float_format="%.3f"))
```

## BOSS giants

**Parent.** The SDSS-V carton mwm_magcloud_rgb_boss ([Nidever et al. 2026](https://doi.org/10.3847/1538-3881/ae7c76)), a carton
being one targeting category of SDSS-V, selected its targets from *Gaia* DR3. We re-implement it from the public target-selection
code with the cuts of Table 2.1: a magnitude limit, a colour limit, two proper-motion cuts, a parallax cut and a polygon in the
dereddened colour–magnitude diagram. The carton dereddens with the reddening $E(B-V)$ of the
[Schlegel et al. (1998)](https://doi.org/10.1086/305772) map, scaled by 0.07 where $E(B-V)$ ≥
0.1 within 4.5° of the LMC (where the map traces LMC dust emission), and $A_V$ =
0.86 × 3.1 $E(B-V)$, with $A_V$ the extinction in the $V$ band. Every cut needs finite photometry and
astrometry, so a star without a five-parameter astrometric solution is not in the parent; no cut is made on RUWE. Within
12°, 558 909 stars pass, and 100 per cent of them are members of the
carton in the public SDSS-V target database: the re-implementation reproduces the carton. SDSS-V observed 81 170 of
them and obtained a good spectrum for 58 198: BOSSNet flag unset, finite [Fe/H] and velocity, S/N ≥
16 and a velocity error below 10 km/s, the highest-S/N spectrum of each star. The giant
and velocity cuts of the samples chapter are not part of this definition; they remove non-members after weighting.

The parent is complete (A1) only if *Gaia* DR3 holds every LMC giant inside the selection with an astrometric solution that passes
the proper-motion window. Two tests bound the loss. APOGEE targeted LMC18–35 without astrometry: of its 2 908
good-spectrum giants there with LMC velocities, 0.3 per cent fail the carton's astrometric cuts, and
2.6 per cent of the 3 073 in the crowded fields LMC1–17. In the dusty centre the parent lacks
1.1 per cent of the stars that the conversion of the stellar-mass chapter expects. We do not correct for
either; both make the masses of the inner few degrees low by a few per cent.

**Selection groups.** Fibres did not go to the parent at random. Some sky areas were observed more often than others, and parent
stars that also belonged to other cartons, chiefly the distant halo K giants and the metal-poor candidates selected from *Gaia* XP
spectra, had higher priority. A selection group therefore combines the fibre-priority class of the star (the carton alone, K giant,
XP metal-poor candidate or other), a pixel of the HEALPix equal-area tessellation of the sphere
([Górski et al. 2005](https://doi.org/10.1086/427976)) at $N_{\rm side}$ = 64 (0.84 deg²)
and a $G$ bin (edges 12, 15.5, 16.25, 16.75, 17.1, 17.5). A group with fewer than 50 parent stars merges with its
neighbours at the next coarser level (no $G$ bins, then $N_{\rm side}$ = 32, 16 and
8, then the whole class). There are 1 198 groups, and 95 per cent
of the parent sits at the finest level. In a group $s$ with $N_{{\rm P},s}$ parent stars, $N_{{\rm O},s}$ observed and
$N_{{\rm G},s}$ good spectra, assumption A2 makes $N_{{\rm P},s}/N_{{\rm O},s}$ the weight of an observed star. If failures also
ignored metallicity, $N_{{\rm P},s}/N_{{\rm G},s}$ would be the weight of a good spectrum.

**Pass factor.** Failures do depend on metallicity: the weak lines of faint metal-poor giants fail more often, and more often
still in the crowded centre. We measure this with a metallicity known for observed and unobserved stars alike, the *Gaia* XP
estimate [M/H]$_{\rm XP}$ of [Andrae et al. (2023)](https://doi.org/10.3847/1538-4365/acd53e) ([M/H]: the overall metal abundance
relative to the Sun's, in dex), kept only for cool giants (XP $T_{\rm eff}$ < 5500 K, $\log g$ <
3.5), where it is reliable ([Zhang et al. 2024](https://doi.org/10.1093/mnras/stae1887));
89 per cent of the parent has one. Pass groups $c$ combine $G$ (edges 12, 16, 16.75, 17.1, 17.5),
[M/H]$_{\rm XP}$ (edges -1.75, -1.5, -1.25, -1, -0.5, and a bin for stars without [M/H]$_{\rm XP}$) and three rings ($\rho$ =
0–3, 3–6 and 6–12°). The pass
factor

$$
f_c=\frac{\sum_{i\in{\rm O}\cap c}N_{{\rm P},s(i)}/N_{{\rm O},s(i)}}{\sum_{i\in{\rm G}\cap c}N_{{\rm P},s(i)}/N_{{\rm G},s(i)}}
$$

is the weighted number of observed stars (set O) over the weighted number of good spectra (set G) in the pass group, $s(i)$ being
the selection group of star $i$, and the weight of a good spectrum $i$ is

$$
w_i=\frac{N_{{\rm P},s(i)}}{N_{{\rm G},s(i)}}\,f_{c(i)}.
$$

The factor is close to unity at $G$ < 16 and at [M/H]$_{\rm XP}$ > -1.25. For
[M/H]$_{\rm XP}$ < -1.75 at 17.1 < $G$ < 17.5 it is 2.0 ±
1.8, 1.8 ± 0.9 and 1.3 ± 0.5 in the
three rings (bootstrap over $N_{\rm side}$ = 16 pixels). A factor common to all rings closes the whole sky
but not the rings: it leaves the share of stars below [M/H]$_{\rm XP}$ = -1.5 at 0.62 of the parent's within
1.5° and 1.27 beyond 6°. The pass factor puts back every failure within
a pass group whatever its cause (A3); it cannot follow a dependence on true metallicity at fixed [M/H]$_{\rm XP}$, or on position
within a ring.

**Coverage.** Groups with at least 3 good spectra are covered: 1 069 groups holding
522 492 parent stars (93.5 per cent). The other 36 417
(6.5 per cent), 67 per cent of them within 3°, are in uncovered
groups. *Gaia* counts these stars, so no model of the spatial density is needed; only their metallicities are unknown. The coverage
factor

$$
c_i=\frac{N_{{\rm P},r(i)}}{N_{{\rm P,cov},r(i)}}
$$

is the ratio of all to covered parent stars in the cell $r$ of ring and $G$ bin (edges 12, 15.5, 16.25, 16.75, 17.1, 17.5) of star $i$; it is
1.04–1.10 per ring and reaches 1.57 for $G$ < 15.5
within 3°. It assumes A4. The holes in the BOSS maps of the samples chapter are mostly not uncovered groups but
covered groups with few spectra: the 73 groups where fewer than 1 per cent of the
parent gave a good spectrum hold 33.4 per cent of the parent with 893 spectra, at a
median $\rho$ of 1.3°, and weights reach 1 477. These groups are unbiased under A2 and A3, and
their noise enters the bootstrap.

**Non-members.** Of the weighted good spectra, 1.1 per cent fail the giant or velocity cuts
(0.8–1.8 per cent per ring). Under A2 the survey observed them at the rate of the members
of their group, so removing them after weighting leaves the weighted members unbiased: the weights add up to the covered parent, and
those of the members to the covered parent less its non-members. A non-member whose spectrum fails is put back by the pass factor
onto the stars of its pass group (A5); dwarfs are 0.3 per cent of the good spectra, which bounds the weight this
can move.

**Tests.** Figure 2.1 (b) compares the share of each [M/H]$_{\rm XP}$ bin in the spectra with that in the covered
parent. The unweighted spectra over-represent the stars below [M/H]$_{\rm XP}$ = -1.25 by
38–78 per cent, because the higher-priority cartons favoured them, and under-represent the
most metal-rich by 44 per cent. With the final weights the shares agree: the fractions below
[M/H]$_{\rm XP}$ = -1, -1.5 and -2 are recovered to 0.0018 ± 0.0032, -0.0003 ±
0.0012 and 0.0002 ± 0.0002, against -0.0080, -0.0029 and
-0.0003 with the selection-group weights alone. Within a pass group this agreement holds by construction, so the figure
tests A2 across groups, for the 89 per cent of stars with [M/H]$_{\rm XP}$. In rings of
0–1.5, 1.5–3, 3–6 and
6–12°, the weighted share below [M/H]$_{\rm XP}$ = -1.5 is 0.82,
0.92, 1.03 and 1.07 of the parent's. The deficit within
1.5° comes from the sparsely sampled central groups, whose few spectra hold too few metal-poor stars. Pass factors
measured on half of the sky (even $N_{\rm side}$ = 8 pixels) and applied to the other half leave that half
short of 0.0040 of its stars below -1.5, 22 per cent of the parent fraction of
0.018 (0.0053 without pass factors): the failure rate varies on scales finer than our rings,
and this test sets the size of that systematic. A mock on the real design, with known metallicities and failures that depend on
metallicity, recovers 0.97 ± 0.09 of the stars below [Fe/H] = -1.5 and 0.88 ±
0.23 below -2; per ring, with the coverage factor and failures twice as frequent within 3°,
it recovers 0.88 ± 0.20, 1.01 and 1.00 below -1.5 (see the
checks below): within 3° the S/N cut leaves many groups with few spectra. The metallicities of the uncovered
parent match those of covered stars in the same rings and $G$ bins to 2 per cent below [M/H]$_{\rm XP}$ = -1 and
5 per cent below -1.5 (A4), and requiring two or five good spectra per covered group instead of three changes the
weighted members by -1.4 and 0.0 per cent. Among the stars with APOGEE spectra, those whose BOSS
spectra the S/N cut rejects have the APOGEE metallicities of those it keeps in the same pass groups (mean difference
-0.018 (+0.024, −0.026) dex, 60 stars), as A3 requires.

### The BOSS weights

`scripts/boss_selection.py` joins the parent with its observed flags, carton memberships and XP metallicities (built in the
samples chapter), forms the selection groups and the pass groups and computes the weights $w_i$. It also runs the closure tests
of Figure 2.1 (b): over the whole sky with a bootstrap over HEALPix pixels, on the held-out half of the sky and per
ring, and it records the pass factor of each pass group with its bootstrap error. It writes `data/boss_weights.parquet` and
`results/boss_selection.json`.

<details><summary>About <code>scripts/boss_selection.py</code></summary>

```text
BOSS selection function: weights for the good spectra of the eligible parent of mwm_magcloud_rgb_boss.

Inputs: data/boss_parent.parquet (eligible), data/boss_observed.parquet (observed, spectrum_ok, giant, BOSSNet feh),
data/boss_cartons.parquet (carton memberships; SDSS-V XP [M/H], kept as mh_xp_sdss for comparison), data/xp_andrae.parquet
(XP [M/H] of Andrae et al. 2023 for cool giants, scripts/build_xp_andrae.py).
Class: lmcold.magcloud.carton_class. Selection groups: HEALPix (nested) x G bin x class, pooled from fine to coarse within the class
(lmcold.groups.assign, NMIN eligible stars): (nside 64, G) -> (64) -> (32) -> (16) -> (8) -> class.
Weights (lmcold.selection): w_group = N_elig / N_good per selection group; w = w_group x pass factor of the star's pass group
(G x XP [M/H] x ring, selection.boss_pass_groups), which puts back the failed spectra: weak-lined metal-poor stars fail more often at faint G, and
more often in the crowded centre (a pass factor common to all rings under-corrects at rho < 3 deg and over-corrects beyond 6 deg).
Checks (results/boss_selection.json):
  C1 closure: w-weighted good spectra minus covered eligible stars, fractions of XP [M/H] below -1, -1.5, -2, for w and
     w_group; errors from a bootstrap over nside-16 pixels that recomputes all counts and factors (NBOOT).
  H  held-out sky: pass factors from the even nside-8 pixels applied to the odd ones (with their own group counts); closure there.
  R  closure per ring (0-1.5, 1.5-3, 3-6, 6-12 deg): w-weighted good spectra over covered eligible stars, fractions of XP [M/H]
     below -1 and -1.5; for the final weights and for a pass factor common to all rings (selection.pass_groups).
  P  pass factor per pass group with its bootstrap error (quoted in the text).
Writes data/boss_weights.parquet (one row per eligible star).

sources.yaml: reads release_boss_parent, release_boss_observed, release_boss_cartons, release_xp_andrae;
writes release_boss_weights. Output columns: source_id, ra, dec [deg], g [mag], cls (carton class), group (selection-group
label 'level:key', lmcold.groups.assign), observed, good (observed and spectrum_ok), giant, w (final weight; 0 for stars that are
not good spectra or lie in uncovered groups), w_group (N_elig / N_good), pass_factor (NaN where w_group = 0), mh_xp (Andrae+23,
cool giants only) and mh_xp_sdss [dex], feh (raw BOSSNet [dex]). The weights serve figures/paper_selection.png (b), the
selection checks in the appendix and the mass estimate (build_giant_mass.py).

Run: .venv/bin/python scripts/boss_selection.py   (after build_boss_observed.py, build_boss_cartons.py, build_xp_andrae.py)
```
</details>

*Shown, not run here: the BOSS parent tables of the samples chapter, including data/boss_cartons.parquet and data/xp_andrae.parquet, which are not in Zenodo bundle 1. The cells below are the script itself; run it from the repository with* `python scripts/boss_selection.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import json

import healpy as hp
import numpy as np
import pandas as pd

from lmcold import magcloud, selection
from lmcold.constants import LMC
from lmcold.footprint import angular_distance
from lmcold.sources import DATA, ROOT
from lmcold.groups import assign, bin_index
```

#### Settings
Pooling threshold of the selection groups, bootstrap size, $G$ bins, pooling levels and closure thresholds.

```python
NMIN = 50  # minimum number of eligible stars in a selection group at a given level of pooling
NBOOT = 200  # bootstrap draws over nside-16 HEALPix pixels
GEDGES = [12.0, 15.5, 16.25, 16.75, 17.1, 17.5]  # G bins [mag] of the finest selection groups (to the carton limit 17.5)
LEVELS = [(64, True), (64, False), (32, False), (16, False), (8, False)]  # (HEALPix nside, split by G?), finest first
CUTS = (-1.0, -1.5, -2.0)  # XP [M/H] thresholds [dex] of the closure checks
```

#### Load the data
The eligible parent (`data/boss_parent.parquet`), its observed and
good-spectrum flags (`data/boss_observed.parquet`), carton memberships (`data/boss_cartons.parquet`)
and XP [M/H] (`data/xp_andrae.parquet`), joined on the Gaia source_id.

```python
# One row per eligible parent star; xp_andrae is a left join (stars without Andrae+23 [M/H] get NaN mh_xp).
c = (pd.read_parquet(DATA / "boss_parent.parquet", columns=["source_id", "ra", "dec", "g", "eligible"]).query("eligible")
     .merge(pd.read_parquet(DATA / "boss_observed.parquet"), on="source_id", validate="1:1")
     .merge(pd.read_parquet(DATA / "boss_cartons.parquet"), on="source_id", validate="1:1").rename(columns={"mh_xp": "mh_xp_sdss"})
     .merge(pd.read_parquet(DATA / "xp_andrae.parquet", columns=["source_id", "mh_xp"]), on="source_id", how="left", validate="1:1"))
```

#### Classes and selection groups
The fibre-priority class of each star, and the selection groups pooled from
fine to coarse until each holds at least NMIN parent stars.

```python
c["cls"] = magcloud.carton_class(c.cartons)
c["good"] = c.observed & c.spectrum_ok
# HEALPix angles: colatitude theta = 90 deg - dec, longitude phi = ra (equatorial pixelisation).
theta, phi = np.radians(90 - c.dec.to_numpy()), np.radians(c.ra.to_numpy())
gbin = pd.Series(bin_index(c.g, GEDGES), c.index).astype(str)
# Keys finest first: class_pix64_Gbin, class_pix64, class_pix32, class_pix16, class_pix8, class.
keys = [c.cls + "_" + pd.Series(hp.ang2pix(nside, theta, phi, nest=True), c.index).astype(str) + ("_" + gbin if by_g else "")
        for nside, by_g in LEVELS] + [c.cls]
c["group"] = assign(keys, NMIN)
s = pd.factorize(c.group)[0]  # integer selection-group index per star
```

#### Rings, pass groups and bootstrap pixels
The arrays the weights and checks use.

```python
obs, good, mh = c.observed.to_numpy(), c.good.to_numpy(), c.mh_xp.to_numpy()
xp = np.isfinite(mh)  # star has an Andrae+23 XP [M/H] (the closure checks use these only)
# Ring index 0-3 for 0-1.5, 1.5-3, 3-6, >= 6 deg from the LMC centre (check R).
rr = np.digitize(angular_distance(c.ra.to_numpy(), c.dec.to_numpy(), LMC["ra0"], LMC["dec0"]), [1.5, 3.0, 6.0])
# Pass group: G bin x XP [M/H] bin x ring (0-3, 3-6, >= 6 deg), selection.boss_pass_groups.
pg = selection.boss_pass_groups(c.g, mh, c.ra, c.dec)
npg = pg.max() + 1
pix8, pix16 = hp.ang2pix(8, theta, phi, nest=True), hp.ang2pix(16, theta, phi, nest=True)
```

#### Helpers
`final` gives the weights, `closure` compares the weighted XP [M/H] fractions with those of the covered
parent, `factor_per_pass_group` returns the pass factors; all take per-star multiplicities for the bootstrap.

```python
def final(m):
    """Final weight per star, N_elig / N_good x pass factor, for multiplicities m.

    Parameters
    ----------
    m : numpy.ndarray
        Per-star multiplicity (ones for the data, bootstrap counts, or a 0/1 mask).

    Returns
    -------
    numpy.ndarray
        Weight per star; 0 unless the star is a good spectrum in a covered selection group.
    """
    # Group weight N_elig / N_good (0 outside covered groups) times the pass factor of the star's pass group.
    return selection.group_weights(s, good, m) * selection.pass_factor(s, obs, good, pg, m)


def closure(w, m):
    """w-weighted good spectra minus covered eligible stars: fractions of XP [M/H] below CUTS (multiplicities m).

    Parameters
    ----------
    w : numpy.ndarray
        Weight per star.
    m : numpy.ndarray
        Per-star multiplicity.

    Returns
    -------
    numpy.ndarray
        One difference of fractions per threshold in CUTS (0 = perfect closure). The reference is every eligible star with XP
        [M/H] in a selection group holding >= NMIN_GOOD good spectra (covered groups).
    """
    # Weighted sample: good spectra with a weight and an XP [M/H].
    sel = (w > 0) & xp
    # Reference: eligible stars with XP [M/H] in covered groups (>= NMIN_GOOD good spectra, counted with multiplicities m).
    ref = (np.bincount(s, m * good)[s] >= selection.NMIN_GOOD) & xp
    # Weighted fraction below x minus the reference fraction below x, for each x in CUTS [dex].
    return np.array([np.sum((m * w)[sel & (mh < x)]) / np.sum((m * w)[sel]) - np.sum(m[ref & (mh < x)]) / np.sum(m[ref])
                     for x in CUTS])


def factor_per_pass_group(m):
    """Pass factor of each pass group, for multiplicities m.

    Parameters
    ----------
    m : numpy.ndarray
        Per-star multiplicity.

    Returns
    -------
    numpy.ndarray
        Length npg; NaN for pass groups with no stars.
    """
    # Per-star pass factor, then scattered to one value per pass group (all stars of a group share it).
    f = selection.pass_factor(s, obs, good, pg, m)
    out = np.full(npg, np.nan)
    out[pg] = f
    return out
```

#### Weights
Group weight, pass factor and final weight of every eligible star.

```python
# Weights of the data (all multiplicities 1).
ones = np.ones(len(c))
c["w_group"] = selection.group_weights(s, good, ones)
c["pass_factor"] = np.where(c.w_group > 0, selection.pass_factor(s, obs, good, pg, ones), np.nan)
c["w"] = final(ones)  # final weight = w_group x pass factor
```

#### Bootstrap
Redraw the HEALPix pixels and recompute the closure and the pass factors.

```python
# Bootstrap over nside-16 pixels: m = number of times a star's pixel is drawn; counts and factors are recomputed per draw.
# Each row of boot: closure of final weights (3), closure of group weights (3), pass factor per pass group (npg).
rng = np.random.default_rng(1)  # fixed seed for reproducibility
upix = np.unique(pix16)  # occupied nside-16 pixels, sorted
boot = []
for _ in range(NBOOT):
    m = np.bincount(np.searchsorted(upix, rng.choice(upix, len(upix))), minlength=len(upix))[np.searchsorted(upix, pix16)].astype(float)
    boot.append(np.concatenate([closure(final(m), m), closure(selection.group_weights(s, good, m), m), factor_per_pass_group(m)]))
boot = np.array(boot)  # shape (NBOOT, 6 + npg)
err = boot.std(0)  # bootstrap error of every entry of a row
```

#### Checks and outputs
Closure overall, on the held-out half of the sky and per ring, and the pass factors; written to
`data/boss_weights.parquet` and `results/boss_selection.json`.

```python
# Held-out sky: even/odd nside-8 pixels (nested numbering); pass factors from the even half, applied to the odd half.
even = (pix8 % 2 == 0).astype(float)
w_odd = selection.group_weights(s, good, 1 - even)
# Check results; each entry is described in the module docstring (C1, H, R, P).
tests = dict(
    # C1: closure for the final weights (error columns 0-2) and the group weights alone (error columns 3-5).
    C1_closure={k: {f"<{x}": dict(diff=round(float(d), 4), err=round(float(e), 4)) for x, d, e in zip(CUTS, v, err[i:i + 3])}
                for (k, v), i in ((("final", closure(c.w.to_numpy(), ones)), 0), (("group", closure(c.w_group.to_numpy(), ones)), 3))},
    # H: odd half weighted by its own group counts times the even-half pass factors.
    H_held_out=dict(final=np.round(closure(w_odd * factor_per_pass_group(even)[pg], 1 - even), 4).tolist(),
                    group=np.round(closure(w_odd, 1 - even), 4).tolist()),
    # Per ring: weighted fraction below x divided by the fraction among covered eligible stars (1 = closure).
    # 'common' uses pass groups without rings (selection.pass_groups).
    R_ring_closure={lab: {f"{lo}-{hi}": {f"<{x}": round(float(np.sum(w[(w > 0) & xp & (rr == k) & (mh < x)]) / np.sum(w[(w > 0) & xp & (rr == k)])
                                                           / np.mean(mh[(np.bincount(s, good)[s] >= selection.NMIN_GOOD) & xp & (rr == k)] < x)), 3)
                                         for x in (-1.0, -1.5)} for k, (lo, hi) in enumerate(zip([0, 1.5, 3, 6], [1.5, 3, 6, 12]))}
                    for lab, w in (("final", c.w.to_numpy()),
                                   ("common", c.w_group.to_numpy() * selection.pass_factor(s, obs, good, selection.pass_groups(c.g.to_numpy(), mh), ones)))},
    # P: pass factor per pass group, bootstrap error (columns 6 onward), and good spectra in covered groups per pass group.
    P_pass_factor=dict(g_edges=selection.PG_EDGES, xp_edges=selection.XP_EDGES, ring_edges=selection.RING_EDGES, index="(G bin x XP bin) x ring + ring", factor=np.round(factor_per_pass_group(ones), 3).tolist(),
                       err=np.round(err[6:], 3).tolist(), n_good=np.bincount(pg[c.w_group > 0], minlength=npg).tolist()),
    # Counts: eligible, good, in covered groups, sums of weights, stars with XP [M/H], stars per class.
    summary=dict(eligible=len(c), good=int(good.sum()), covered=int((np.bincount(s, good)[s] >= selection.NMIN_GOOD).sum()),
                 sum_w_strata=round(float(c.w_group.sum()), 1), sum_w=round(float(c.w.sum()), 1), with_xp=int(xp.sum()),
                 classes=c.cls.value_counts().to_dict()))
# One row per eligible star (columns in the module docstring), then the checks as JSON.
c[["source_id", "ra", "dec", "g", "cls", "group", "observed", "good", "giant", "w", "w_group", "pass_factor", "mh_xp", "mh_xp_sdss",
   "feh"]].to_parquet(DATA / "boss_weights.parquet", index=False)
(ROOT / "results" / "boss_selection.json").write_text(json.dumps(tests, indent=1))
print(json.dumps({k: v for k, v in tests.items() if k != "P_pass_factor"}, indent=1))
```

## APOGEE giants

**Parent.** APOGEE-2 (the second phase of APOGEE) selected its LMC giants from 2MASS photometry
([Skrutskie et al. 2006](https://doi.org/10.1086/498708)) in a polygon of $J-K_s$ against $H$
([Zasowski et al. 2017](https://doi.org/10.3847/1538-3881/aa8df9); [Nidever et al. 2020](https://doi.org/10.3847/1538-4357/ab7305);
[Santana et al. 2021](https://doi.org/10.3847/1538-3881/ac2cbc)): bright giants (12.35 ≤ $H$ <
12.8) at random, and fainter ones (12.8 ≤ $H$ ≤ $H_{\rm lim}$) brightest first. $H_{\rm lim}$
is 14.5 (14.9 in LMC2 and LMC12) in LMC1–17 and 15.3 in LMC18–35, and plate
design dropped stars with a neighbour at least 2 mag brighter in $H$ within 6″. We rebuild
the candidates in each field from the public targeting files: 284 441 stars inside the polygon within
0.95° of a field centre, of which the bright-neighbour rule removes 922. The polygons contain
99.86 per cent of the stars APOGEE targeted in LMC1–17 and 100 per cent of those in LMC18–35.
In LMC1–17 APOGEE also required a giant flag from Washington and DDO51 photometry (W+D; two filters whose combination separates
giants from dwarfs), which DR17 does not publish. In its place, and in all 35 fields, we require the
astrometric cuts of the BOSS carton. They leave 193 418 candidates; in LMC18–35 they remove 43
per cent of the candidates, nearly all of them Milky Way dwarfs, and 0.3 per cent of the giants with LMC
velocities.

**Weights.** Selection groups are a field, an annulus (distance from the field centre $\rho_{\rm field}$ <
0.8° or 0.8–0.95°), the bright or faint class, $H$ in
0.1-mag bins for the faint class, and 5 bins of position across the colour range of the
polygon. A good spectrum has the ASPCAP flag STAR_BAD unset, a finite [Fe/H], S/N ≥ 30 and is not a
duplicate; as for BOSS, the giant cut comes after weighting. APOGEE observed 6 387 candidates and obtained
5 918 good spectra. A bright group is covered with at least 3 good spectra, a faint group
with at least 2 and an observed fraction of at least 0.2 (which drops the thin bins at each
field's fibre limit), and any group with at least a fraction 0.5 of its candidates good; the covered groups hold
16 390 candidates. The weights follow the equation for $w_i$ above, with pass groups of $H$ (edges
12.35, 13.3, 14.3, 15.3) and [M/H]$_{\rm XP}$: ASPCAP fails (STAR_BAD set or no [Fe/H]) for 21 per cent of the
bright observed stars with S/N ≥ 30 and [M/H]$_{\rm XP}$ < -1.5 against 1.6 per cent of those with
[M/H]$_{\rm XP}$ > -1. APOGEE fields are small, so we use no coverage factor across the sky; the stellar-mass
chapter uses APOGEE only in field cores. Of the weighted good spectra, 0.1 per cent (LMC1–17) and
0.4 per cent (LMC18–35) fail the velocity cut.

**Tests.** Within each group class the weighted mean $J-K_s$ of the good spectra matches that of the covered candidates to
0.0022 mag, and the weighted fraction of stars below [Fe/H] = -1.5 changes by at most 0.0004 between S/N
thresholds of 30, 50, 70. Figure 2.1 (c) shows the test against [M/H]$_{\rm XP}$. In LMC18–35 the weighted share
below [M/H]$_{\rm XP}$ = -1.5 is 1.17 of the parent's (1.10 before the pass factor); in
LMC1–17 it is 1.61 (1.41 before; field-bootstrap error 0.34). The unpublished giant flag, which our astrometric stand-in
does not reproduce, is the likely cause. We do not correct for it and use APOGEE as a check of the BOSS masses, not as their source.

### The APOGEE weights

`scripts/apogee_selection.py` keeps one row per candidate, forms the selection groups, computes the weights with pass groups of
$H$ and [M/H]$_{\rm XP}$, and runs the tests quoted above: the share of the APOGEE targets inside the rebuilt polygons, the
weighted $J-K_s$ colour, the closure in [M/H]$_{\rm XP}$ of Figure 2.1 (c) and the comparison of S/N thresholds. It
writes `data/apogee_weights.parquet` and `results/apogee_selection.json`.

<details><summary>About <code>scripts/apogee_selection.py</code></summary>

```text
APOGEE selection function: weights for the quality stars of the LMC1-35 RGB targeting (data/apogee_parent.parquet).

Candidates: astro_ok and no bright neighbour; a star in two fields keeps the row of the field that observed it (else the first).
Selection groups: field x zone (r < 0.8 deg | 0.8-0.95 deg; the 2017 designs stop at 0.8 deg) x class (BrtRGB one H bin; FntRGB 0.1-mag H
bins, filled brightest first) x colour bin (x = position across the polygon, edges XEDGES).
Quality: observed and lmcold.giants.apogee_spectrum_ok with S/N >= SNR_MIN, giant or not (lmcold.apogee_target.weights); the giant
cut and the velocity window are applied after weighting (no VSCATTER cut: it fails a quarter of the stars below [Fe/H] = -2).
  w = N_cand / N_quality x pass factor for quality stars in covered groups: BrtRGB groups with >= 3 quality stars; FntRGB groups
  with >= 2 and f_obs >= 0.2 (drops the thin bins at each field's fibre limit); any group with f_q >= 0.5 and >= 1 quality star.
  Pass factor (lmcold.selection, as for BOSS): pass groups H bin (PG_H_EDGES) x XP [M/H] of Andrae et al. 2023; ASPCAP fails
  (STAR_BAD, no FE_H) for 11% of the bright XP < -1.5 stars against 1.6% of XP > -1, at S/N ~ 150.
Checks (results/apogee_selection.json):
  T1 polygon reproduction: LMC1-17 targets with the W+D bit (APOGEE2_TARGET1 bit 7) inside the parent polygons (>= 99.5%);
     LMC18-35 Magellanic targets with H >= 12.35 and 0.5 < J - Ks < 1.3 inside them (>= 99%).
  T5 the weighted mean J - Ks of quality stars equals the mean of the covered candidates to < 0.005 mag (~0.03 dex in [Fe/H]),
     per group (W+D fields or not) x class.
  T7 XP closure: weighted quality stars / covered candidates, fractions of XP [M/H] below -1 and -1.5, with and without the pass
     factor, for the W+D fields and LMC18-35 separately. The W+D fields give 1.41 +- 0.33 below -1.5 without the pass factor
     (field bootstrap): the DDO51 giant flag, which is not reproduced, may favour metal-poor giants. Not corrected; a stated
     systematic of the APOGEE metal-poor counts.
  T6 S/N neutrality: weighted fractions of members (150 < v < 420 km/s) below [Fe/H] = -1.5 and -2 for S/N >= 30, 50, 70, in
     the groups covered at all three; they must agree within the group-bootstrap error of the first (0.0023). A lower S/N cut
     only adds coverage (faint groups).
Writes data/apogee_weights.parquet (one row per candidate star).

sources.yaml: reads release_apogee_parent and apogee_dr17_allvisit (for T1), plus data/xp_andrae.parquet
(scripts/build_xp_andrae.py, not registered); writes release_apogee_weights. Output columns: field, apogee_id, source_id,
ra, dec [deg], j, h, k [mag], cls, group, observed, quality, giant, covered, w, feh [dex, ASPCAP], v [km/s, heliocentric],
mh_xp [dex]. Note: lmcold.selection.pass_factor counts only groups with >= NMIN_GOOD = 3 quality stars, while the APOGEE
coverage rule above also admits FntRGB groups with 2 and any group with 1 (f_q >= 0.5). The weights serve
figures/paper_selection.png (c) and the APOGEE mass (build_giant_mass.py).

Run: .venv/bin/python scripts/apogee_selection.py   (after build_apogee_parent.py and build_xp_andrae.py)
```
</details>

*Shown, not run here: data/apogee_parent.parquet, data/xp_andrae.parquet and the APOGEE DR17 allVisit file (scripts/build_apogee_parent.py, scripts/build_xp_andrae.py, scripts/fetch_targeting.py). The cells below are the script itself; run it from the repository with* `python scripts/apogee_selection.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import json

import numpy as np
import pandas as pd
from astropy.io import fits

from lmcold import apogee_target as at
from lmcold import giants, selection
from lmcold.sources import DATA, ROOT, path
```

#### Settings
Colour bins of the selection groups, the S/N cut of a quality spectrum and the $H$ bins of the pass groups.

```python
XEDGES = [0.4, 0.6, 0.8, 0.9]  # edges of the relative colour x (0 = blue edge, 1 = red edge of the polygon): 5 colour bins
SNR_MIN = 30  # APOGEE S/N cut of a quality spectrum (check T6 compares 30, 50, 70)
PG_H_EDGES = [12.35, 13.3, 14.3, 15.3]  # H bins [mag] of the pass groups, from the BrtRGB bright limit to the faintest maglim
```

#### Load the data
The APOGEE parent `data/apogee_parent.parquet`
(`scripts/build_apogee_parent.py`). The XP metallicities `data/xp_andrae.parquet` are read in the
pass-group step and the allVisit file in test T1.

```python
# one row per (field, star) of the LMC1-35 RGB polygons; columns include j, h, k [mag], cls, maglim, rfield [deg], observed, snr
p = pd.read_parquet(DATA / "apogee_parent.parquet")
```

#### Candidates
Polygon stars with the astrometric cuts and no bright neighbour, one row per star.

```python
# Candidates; sorting puts a star's observed row first, so drop_duplicates keeps the field that observed it.
c = p[p.astro_ok & ~p.nbr].sort_values(["apogee_id", "observed"], ascending=[True, False]).drop_duplicates("apogee_id").copy()
# LMC1-17: fields whose targeting also used the Washington + DDO51 giant flag.
# field names are 'LMC<n>'; characters from index 3 give n
c["wd_field"] = c.field.str[3:].astype(int) <= 17
```

#### Selection groups
Field, zone, class, $H$ bin and position across the colour range of the polygon.

```python
# blue and red J - Ks [mag] edges of the targeting polygon at each star's H (lmcold.apogee_target.sides)
blue, red = at.sides(c.h.to_numpy(), c.maglim.to_numpy())
# x: relative colour across the polygon, 0 at the blue edge and 1 at the red edge
c["x"] = (c.j - c.k - blue) / (red - blue)
# (12.8 is the BrtRGB/FntRGB boundary of the polygons, lmcold.apogee_target)
# H bin: 0 for BrtRGB; FntRGB 1, 2, ... for 0.1-mag bins from H = 12.8.
hbin = np.where(c.cls == 1, 0, 1 + np.floor((c.h - 12.8) / 0.1).clip(0)).astype(int)
# Selection group label: field_zone_class_Hbin_colourbin, zone 0 for r < 0.8 deg and 1 for 0.8-0.95 deg.
c["group"] = (c.field + "_" + (c.rfield >= 0.8).astype(int).astype(str) + "_" + c.cls.astype(str) + "_" + hbin.astype(str) + "_"
             + np.digitize(c.x, XEDGES).astype(str))
```

#### Giant flag, XP metallicity and pass groups
The giant flag of stars with spectra, the XP [M/H] of Andrae et al.
(2023) and the pass groups ($H$ bin and XP [M/H]).

```python
s = c[c.snr.notna()]  # stars with an allStarLite row
# is_giant evaluated on stars with spectra; stars without one get giant = False
c["giant"] = pd.Series(giants.is_giant(s), s.index).reindex(c.index, fill_value=False)
# XP [M/H] [dex] of Andrae et al. 2023 by Gaia source_id; NaN where missing
c["mh_xp"] = c.source_id.map(pd.read_parquet(DATA / "xp_andrae.parquet").set_index("source_id").mh_xp).astype(float)
# pass-group label of each candidate: H bin (PG_H_EDGES) x XP [M/H] bin (lmcold.selection.pass_groups)
pg = selection.pass_groups(c.h.to_numpy(), c.mh_xp.to_numpy(), PG_H_EDGES)
```

#### Weights
Quality and coverage flags and the weight of each candidate (`lmcold.apogee_target.weights`).

```python
# (quality, covered, w) for a given S/N cut, with or without the pass factor.
weights = lambda c, snr_min, with_pass=True: at.weights(c, snr_min, pg, with_pass)  # noqa: E731
# quality, covered: boolean (N,); w: weight (N,), 0 outside covered quality stars
c["quality"], c["covered"], c["w"] = weights(c, SNR_MIN)
```

#### Test T1: the targeting polygons
Share of the stars APOGEE targeted that lie in the rebuilt parent.

```python
# T1: Magellanic targets (LCO, bit 23 set, bit 22 unset) and whether each is in the parent.
with fits.open(path("apogee_dr17_allvisit"), memmap=True) as h:
    # allVisit table: one row per visit; keep field, star, telescope, targeting flag APOGEE2_TARGET1, H and J - K [mag]
    d = h[1].data
    v = pd.DataFrame({"field": np.char.strip(d["FIELD"].astype(str)), "apogee_id": np.char.strip(d["APOGEE_ID"].astype(str)),
                      "tel": np.char.strip(d["TELESCOPE"].astype(str)), "t1": d["APOGEE2_TARGET1"].astype(np.int64),
                      "h": d["H"].astype(float), "jk": (d["J"] - d["K"]).astype(float)})
# Magellanic targets observed at LCO (the du Pont 2.5-m) in the LMC1-35 fields, one row per (field, star)
v = v[(v.tel == "lco25m") & v.field.isin(at.FIELDS) & giants.bit(v.t1, 23) & ~giants.bit(v.t1, 22)].drop_duplicates(["field", "apogee_id"])
v["in_parent"] = pd.MultiIndex.from_frame(v[["field", "apogee_id"]]).isin(pd.MultiIndex.from_frame(p[["field", "apogee_id"]]))
# W+D targets: bit 7 of APOGEE2_TARGET1
wd = v[giants.bit(v.t1, 7)]
# LMC18-35 targets inside the plain colour-magnitude box of their targeting.
box = v[(v.field.str[3:].astype(int) >= 18) & (v.h >= 12.35) & (v.jk > 0.5) & (v.jk < 1.3)]
# share of each target set inside the parent; pass thresholds 99.5% (W+D) and 99% (LMC18-35), as in the module docstring
tests = {"T1": dict(wd_targets=len(wd), wd_in_parent=round(wd.in_parent.mean(), 4), lmc18_35_box=len(box),
                    lmc18_35_in_parent=round(box.in_parent.mean(), 4))}
tests["T1"]["pass"] = bool(tests["T1"]["wd_in_parent"] >= 0.995 and tests["T1"]["lmc18_35_in_parent"] >= 0.99)
```

#### Test T5: colour
Weighted mean $J-K_s$ of the quality stars against that of the covered candidates.

```python
# T5: weighted mean J - Ks of quality stars minus the unweighted mean of covered candidates [mag].
cov = c[c.covered]
t5 = {}
# two field groups (W+D or not) x two classes (1 = BrtRGB, 2 = FntRGB)
for lab, m in (("LMC1-17", cov.wd_field), ("LMC18-35", ~cov.wd_field)):
    for cl, nm in ((1, "BrtRGB"), (2, "FntRGB")):
        # covered candidates of the group, and its quality stars
        s = cov[m & (cov.cls == cl)]
        q = s[s.quality]
        t5[f"{lab} {nm}"] = round(float(np.average(q.j - q.k, weights=q.w) - (s.j - s.k).mean()), 4)
# pass if every difference is below 0.005 mag
tests["T5_dJK"] = t5 | {"pass": bool(all(abs(x) < 0.005 for x in t5.values()))}
```

#### Test T7: XP closure
Weighted share of stars below two XP [M/H] thresholds over the share among covered
candidates, with and without the pass factor.

```python
# T7: weighted fraction below x over the fraction among covered candidates (1 = closure).
# stars with an XP [M/H]
xp = c.mh_xp.notna()
# per field group and weight set (final, or no pass factor): for x = -1 and -1.5,
# [sum of weights of stars below x / sum of weights] / [fraction below x among covered candidates]
tests["T7_xp_closure"] = {f"{fl} {lab}": {f"<{x}": round(float(np.sum(w[(w > 0) & xp & m & (c.mh_xp < x)]) / np.sum(w[(w > 0) & xp & m])
                                                           / np.mean(c.mh_xp[c.covered & xp & m] < x)), 3) for x in (-1.0, -1.5)}
                          for fl, m in (("LMC1-17", c.wd_field), ("LMC18-35", ~c.wd_field))
                          for lab, w in (("final", c.w.to_numpy()), ("no_pass", weights(c, SNR_MIN, False)[2]))}
```

#### Test T6: S/N cut
Weighted fractions of metal-poor members for three S/N cuts.

```python
# T6: members are giants with 150 < v < 420 km/s; compare S/N cuts on the groups covered under all three.
mem = (c.v > 150) & (c.v < 420)
# (quality, covered, w) for each S/N cut
runs = {snr: weights(c, snr) for snr in (30, 50, 70)}
# groups covered under all three cuts
common = np.logical_and.reduce([cov for _, cov, _ in runs.values()])
t6 = {}
for snr, (q, _, w) in runs.items():
    # quality giants that are members, in the common groups
    m = q & c.giant & mem & common
    # weighted fractions below [Fe/H] = -1.5 and -2 (ASPCAP feh)
    t6[f"snr>={snr}"] = dict(N=int(m.sum()), f_below_m15=round(float(w[m & (c.feh < -1.5)].sum() / w[m].sum()), 4),
                             f_below_m2=round(float(w[m & (c.feh < -2)].sum() / w[m].sum()), 4))
f15 = [r["f_below_m15"] for r in t6.values()]
t6["pass"] = bool(max(f15) - min(f15) < 0.0023)  # the group-bootstrap error of f_below_m15
tests["T6_snr_neutrality"] = t6
# counts of candidates, covered, observed and quality stars, and the sum of the weights
tests["summary"] = dict(candidates=len(c), covered=int(c.covered.sum()), observed=int(c.observed.sum()), quality=int(c.quality.sum()),
                        quality_covered=int((c.quality & c.covered).sum()), sum_w=round(float(c.w.sum()), 1))
```

#### Write the weights
One row per candidate, `data/apogee_weights.parquet`, and the tests,
`results/apogee_selection.json`.

```python
# one row per candidate (columns listed in the module docstring)
c[["field", "apogee_id", "source_id", "ra", "dec", "j", "h", "k", "cls", "group", "observed", "quality", "giant", "covered", "w", "feh",
   "v", "mh_xp"]].to_parquet(DATA / "apogee_weights.parquet", index=False)
# the test results, also printed
(ROOT / "results" / "apogee_selection.json").write_text(json.dumps(tests, indent=1))
print(json.dumps(tests, indent=1))
```

**Mass samples.** The masses of the stellar-mass chapter use the quality cuts of the selection functions, without the kinematic
cuts of the samples chapter, with the giant cut and 150 < $v_{\rm los}$ < 420 km/s
(calibrated velocities for BOSS). They hold 57 557 BOSS and 5 796 APOGEE giants with positive
weights.

### Counts of the mass samples

`scripts/paper_giant_counts.py` counts the giants of the mass samples per [Fe/H] bin, without and with the weights, and compares
the weighted BOSS and APOGEE counts on a common parent, the *Gaia* stars in covered groups of both surveys, with a bootstrap over
HEALPix pixels. Its sample sizes enter Table 2.1; the stellar-mass chapter shows the same comparison.

<details><summary>About <code>scripts/paper_giant_counts.py</code></summary>

```text
Corrected giant counts and an equal-parent APOGEE/BOSS comparison (§2.2/2.3).

Same common parent = Gaia IDs in covered BOSS and APOGEE eligible samples;
NOT just stars with both spectra. Pixel bootstrap uses the same draws for both
surveys, keeping fitted selection weights fixed (conditional counting errors).
Counts are per bin; first and last bins include all [Fe/H]<-2 and >=0.
No isochrone conversion or foreground subtraction is applied to this figure.

Inputs: lmcold.mass_samples.samples(), i.e. the BOSS and APOGEE mass samples (velocity-selected giants with selection weights w;
BOSS [Fe/H] calibrated to the APOGEE scale) from data/boss_weights.parquet (sources.yaml release_boss_weights),
data/apogee_weights.parquet (release_apogee_weights), data/spec_giants.parquet and results/boss_calibration.json, and the
common covered parent (Gaia source_id in a covered BOSS selection group and a covered APOGEE parent).
Method: weighted counts in the ten reporting [Fe/H] bins (lmcold.mass_samples.FEH_EDGES, open ends) per nside-16 HEALPix pixel;
500 multinomial redraws of the pixels (same draws for all four count sets) give the errors.
Outputs:
- results/giant_counts.csv, one row per [Fe/H] bin: feh_lo, feh_hi; boss_raw (unweighted BOSS members), boss_weighted (BOSS
  with w), boss_common and apogee_common (weighted, common parent only), boss_common_raw and apogee_common_raw (unweighted),
  ratio = boss_common / apogee_common with bootstrap ratio_p16, ratio_p84; *_sd: bootstrap standard deviation of each count.
- results/giant_counts.json: sample sizes and the total common-parent BOSS/APOGEE ratio with its 16th-84th percentiles.
- figures/paper_giant_counts.png (not placed in the paper; its content is panel (b) of paper_mass_method, docs/figures.md).
Panels (1 x 3; x: [Fe/H] reporting bin centre, ticks '<-2' ... '>=0'):
(a) y: giants per [Fe/H] bin (log); BOSS covered sample, observed (unweighted, grey) and selection-corrected (blue); vertical
    bars: bootstrap 16th-84th percentiles.
(b) y: giants per bin (log); common parent, corrected BOSS (blue) and corrected APOGEE (orange).
(c) y: corrected BOSS / APOGEE in the common parent with bootstrap 16th-84th percentiles; bins with >= 5 unweighted stars in
    both surveys only.
Run: .venv/bin/python scripts/paper_giant_counts.py   (about 2 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_giant_counts.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

*Shown, not run here: data/spec_giants.parquet, which is not in Zenodo bundle 1 (scripts/build_spec_giants.py); about 2 s. The cells below are the script itself; run it from the repository with* `python scripts/paper_giant_counts.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import healpy as hp
import numpy as np
import pandas as pd
import json

from lmcold.mass_samples import samples, FEH_EDGES, FEH_CENTRES
from lmcold.layout import panel_grid
from lmcold.sources import ROOT
```

#### Load the data
The BOSS and APOGEE mass samples with their selection weights, and the common covered parent
(`lmcold.mass_samples.samples`, from `data/boss_weights.parquet`, `data/apogee_weights.parquet`,
`data/spec_giants.parquet` and `results/boss_calibration.json`).

```python
b,a,common = samples()   # BOSS and APOGEE mass samples (DataFrames with w, feh [dex], ra, dec [deg]) and common-parent Gaia ids
```

#### Four count sets
BOSS unweighted and weighted, and both surveys weighted within the common parent.

```python
bc,ac = b[b.source_id.isin(common)],a[a.source_id.isin(common)]   # BOSS and APOGEE members inside the common parent
# Four count sets (sample, weight per star): BOSS unweighted, BOSS weighted, common-parent BOSS and APOGEE weighted.
sets=[(b,np.ones(len(b))), (b,b.w.to_numpy()), (bc,bc.w.to_numpy()), (ac,ac.w.to_numpy())]
```

#### Counts and pixel bootstrap
Weighted counts per HEALPix pixel and [Fe/H] bin, and their sums over 500
bootstrap draws of the pixels.

```python
# nside-16 HEALPix pixels (~3.7 deg), nested ordering; upix: every pixel occupied by any set, shared by all sets.
pixels=[hp.ang2pix(16,t.ra.to_numpy(),t.dec.to_numpy(),lonlat=True,nest=True) for t,_ in sets]
upix=np.unique(np.concatenate(pixels))
rng=np.random.default_rng(20261026)
# Pixel bootstrap: each of 500 draws gives the number of times every pixel is picked (len(upix) picks); shape (500, n_pix).
draws=rng.multinomial(len(upix),np.ones(len(upix))/len(upix),size=500)
counts,boots=[],[]
for (t,w),pix in zip(sets,pixels):
    j=np.digitize(t.feh,FEH_EDGES)-1   # reporting [Fe/H] bin index 0..9 of each star (open end bins)
    mat=np.zeros((len(upix),len(FEH_CENTRES)))  # weighted counts per pixel x [Fe/H] bin
    np.add.at(mat,(np.searchsorted(upix,pix),j),w)
    counts.append(mat.sum(axis=0))   # total count per [Fe/H] bin
    boots.append(draws@mat)  # (500, n_bins): counts of each bootstrap draw
counts=np.array(counts);boots=np.array(boots)  # (4, n_bins) and (4, 500, n_bins), in the order of `sets`
```

#### Ratios and tables
The BOSS/APOGEE ratio in the common parent with bootstrap percentiles; written to
`results/giant_counts.csv` and `results/giant_counts.json`.

```python
# Unweighted common-parent counts per bin, BOSS and APOGEE.
rawb=np.histogram(bc.feh,FEH_EDGES)[0];rawa=np.histogram(ac.feh,FEH_EDGES)[0]
# Weighted BOSS / APOGEE ratio per bin, and per bootstrap draw (NaN where the APOGEE draw is empty); shape (500, n_bins).
with np.errstate(divide='ignore',invalid='ignore'):
    ratio=counts[2]/counts[3]
    bratio=np.where(boots[3]>0,boots[2]/boots[3],np.nan)
# 16th and 84th percentiles of the bootstrap ratio, only for bins with at least one finite draw (NaN otherwise).
low,high=np.full(len(FEH_CENTRES),np.nan),np.full(len(FEH_CENTRES),np.nan)
has_ratio=np.any(np.isfinite(bratio),axis=0)
low[has_ratio],high[has_ratio]=np.nanpercentile(bratio[:,has_ratio],[16,84],axis=0)
# Table of counts, ratios and bootstrap SDs per [Fe/H] bin (columns in the module docstring).
t=pd.DataFrame({'feh_lo':FEH_EDGES[:-1],'feh_hi':FEH_EDGES[1:],
                'boss_raw':counts[0],'boss_weighted':counts[1],'boss_common':counts[2],'apogee_common':counts[3],
                'boss_common_raw':rawb,'apogee_common_raw':rawa,'ratio':ratio,'ratio_p16':low,'ratio_p84':high})
for i,name in enumerate(['boss_raw','boss_weighted','boss_common','apogee_common']):
    t[name+'_sd']=boots[i].std(axis=0)
t.to_csv(ROOT/'results/giant_counts.csv',index=False)
# Sample sizes and the total common-parent ratio with its bootstrap 16th-84th percentiles.
summary=dict(common_parent=len(common),boss_members=len(b),apogee_members=len(a),boss_common=len(bc),apogee_common=len(ac),
             common_total_ratio=float(counts[2].sum()/counts[3].sum()),
             common_total_ratio_p16_p84=np.percentile(boots[2].sum(axis=1)/boots[3].sum(axis=1),[16,84]).tolist())
(ROOT/'results/giant_counts.json').write_text(json.dumps(summary,indent=2))
```

#### The figure
Counts per [Fe/H] bin before and after the correction, in the common parent, and their ratio.

```python
plt.style.use(ROOT/'src/lmcold/paper.mplstyle')
fig,axes=panel_grid(1,3,width=7.,left=.5,right=.08,hgap=.48,bottom=.45,top=.65)
blue,orange='#2878b5','#d75a20'
# Panels (a) and (b): indices into `sets`, legend labels and colours.
for ax,inds,labels,cols in [(axes[0,0],[0,1],['Observed','Selection corrected'],['0.6',blue]),
                           (axes[0,1],[2,3],['BOSS','APOGEE'],[blue,orange])]:
    for i,label,color in zip(inds,labels,cols):
        # Bootstrap 16th-84th percentile bars and the counts, in bins with a positive count (log axis).
        lo,hi=np.percentile(boots[i],[16,84],axis=0)
        positive=counts[i]>0
        ax.vlines(FEH_CENTRES[positive],lo[positive],hi[positive],color=color,lw=.8)
        ax.plot(FEH_CENTRES[positive],counts[i][positive],'o-',ms=2.5,color=color,label=label,lw=.8)
    ax.set(yscale='log',ylabel='Giants per metallicity bin')
    ax.legend(loc='lower left',bbox_to_anchor=(0,1.1),borderaxespad=0)
# Ratios are shown only where both surveys have >= 5 unweighted common-parent stars and the bootstrap interval is defined.
valid=(rawb>=5)&(rawa>=5)&np.isfinite(low)&np.isfinite(high)
ax=axes[0,2]
# Quantile intervals plotted directly; their midpoint need not equal the data ratio.
ax.vlines(FEH_CENTRES[valid],low[valid],high[valid],color=blue,lw=.8)
ax.plot(FEH_CENTRES[valid],ratio[valid],'o',color=blue,ms=3)
ax.axhline(1,color='0.5',ls='--',lw=.7)
ax.set(ylabel='Corrected BOSS / APOGEE',ylim=(0,max(2.0,float(np.max(high[valid]))*1.05)))
for ax,title in zip(axes[0],['(a) BOSS covered sample','(b) Common parent sample','(c) Common parent sample']):
    ax.set(title=title,xlabel=r'[Fe/H]',xlim=(-2.45,.45),xticks=[-2.25,-1.5,-.5,.25],xticklabels=['<−2','−1.5','−0.5','≥0'])
fig.savefig(ROOT/'figures/paper_giant_counts.png')
print(json.dumps(summary,indent=2))
```

## Checks

### Completeness of the RR Lyrae, and the BOSS mock

Figure 2.2 adds three checks to Figure 2.1. The *Gaia* completeness of RRab falls at bright apparent
magnitudes in the inner 4°: these are blends, brighter than single RRab at the LMC distance, which
*Gaia* does not classify. On the sky, the *Gaia* completeness dips only at the centre, and OGLE's own completeness only at the edges
of its footprint. The BOSS mock keeps the real parent, selection groups, observed flags and the stars with XP metallicities,
assigns true [Fe/H] from the observed distribution, and lets spectra fail more often for faint metal-poor stars. The
selection-group weights alone lose up to 28 per cent of the stars below [Fe/H] = -2; the final weights
recover all bins within their scatter. When observation itself favours metal-poor stars by a factor of 1.3,
which violates the assumption of the weights, the recovered counts below -1.5 are 19 per cent high: the test
in Figure 2.1 (b) would show such a bias, and does not.

The same mock tests the coverage factor. Its truth is now every parent star, including those of uncovered groups, and a second
failure model makes metal-poor spectra fail twice as often within 3°. With the coverage factor,
20 realisations recover 0.88 ± 0.20, 1.01 ±
0.07 and 1.00 ± 0.04 of the stars below [Fe/H] = -1.5 in the rings
0–3, 3–6 and 6–12°; the low and
noisy inner value reflects the many central groups left with few spectra by the S/N cut. Two alternatives do no better. Filling each
gap from covered stars of the same region, $G$ bin and [M/H]$_{\rm XP}$ bin gives 0.90 ±
0.20, 1.01 ± 0.07 and 0.99 ±
0.04; treating the sparsely sampled groups as gaps leaves too few covered stars in the centre, so their parent
is filled from coarser cells and the ring 0–3° gets 1.00–1.37 of its
stars. With the adopted coverage the total counts per ring come out at 0.97, 1.05 and
1.03 in every realisation: the coarse selection groups, up to $N_{\rm side}$ = 8
(7° pixels), straddle the rings and move weight between them, so regional masses are smeared by a few per
cent, while the total is not affected.

<details><summary>About <code>scripts/paper_selection_checks.py</code></summary>

```text
Paper figure (appendix, Sect. 2.2): checks of the selection functions of scripts/paper_selection.py.

RRab, from the overlap of OGLE-IV (own discoveries only: stars from OGLE-LMC-RRLYR-39960 on were added from Gaia) and Gaia DR3:
C_Gaia = fraction of OGLE-own RRab that Gaia also classifies as RRab; C_OGLE = fraction of Gaia RRab inside the OGLE-IV footprint
found by OGLE itself. (a) C_Gaia against mean magnitude in three rings; (b, c) C_Gaia and C_OGLE in 1 deg pixels (>= 5 stars in
both), with the OGLE-IV footprint outline. (d) BOSS truth mock on the real design (scripts/boss_mock.py): weighted count / true
count in bins of true [Fe/H], selection-group weights alone and final weights; mean and sd of the realisations.
Writes figures/paper_selection_checks.png and results/rrab_selection.csv (per pixel).

Inputs: data/rrab.parquet (release_rrab; all RRab within 20 deg, flags ogle_own, in_gaia, OGLE mean I), results/boss_mock.json
(scripts/boss_mock.py; key obs_bias_1.0 = mock without an observation bias).
Panels (1 x 4, square):
(a) x: I + 0.6 mag (~ Gaia G for LMC RRab) [mag], 17.8-20.6, 0.25-mag bins with >= 20 stars; y: Gaia completeness of the
    OGLE-own RRab, binomial errors; rings 0-2, 2-4, 4-12 deg.
(b, c) x (east, increasing to the left), y (north) [deg], +-12.5: C_Gaia and C_OGLE in 1-deg pixels of the orthographic
    projection, colour 0.4-1 (shared bar above both); only pixels inside the OGLE-IV footprint with >= 5 stars found by both;
    grey contour: OGLE-IV footprint.
(d) x: true [Fe/H] [dex] (mock bins); y: recovered / true weighted count, 0.5-1.7; grey: selection-group weights alone, blue:
    final weights; mean and sd over the mock realisations.
Output CSV, one row per 1-deg pixel: x, y [deg] (centre), n_ogle, n_gaia, n_both, C_gaia = n_both / n_ogle, C_ogle =
n_both / n_gaia, C_union = (n_ogle + n_gaia - n_both) / (n_ogle n_gaia / n_both).
Placement: appendix (Sect. 2.2; docs/figures.md). Run: .venv/bin/python scripts/paper_selection_checks.py   (about 3 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_selection_checks.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/raw/ogle4/fields-OGLE-IV-LMC.txt` | public archive | OGLE-IV field centres, LMC (LMC); Udalski et al. 2015, AcA 65, 1 |
| `data/raw/ogle4/fields-OGLE-IV-MBR.txt` | public archive | OGLE-IV field centres, MBR (Magellanic Bridge); Udalski et al. 2015, AcA 65, 1 |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/boss_mock.json` | repository | mock tests of the BOSS selection function; written by `python scripts/boss_mock.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.ndimage import binary_closing, binary_fill_holes
import numpy as np
import pandas as pd

from lmcold import ogle
from lmcold.constants import LMC
from lmcold.footprint import lmc_xy
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

plt.style.use(ROOT / "src" / "lmcold" / "paper.mplstyle")
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"   # line colours (rings of panel a; final weights of panel d)
L = 12.5  # deg, half width of the map panels

#### Load the data
The RRab catalogue `data/rrab.parquet` (Zenodo bundle 1) and the BOSS truth mock `results/boss_mock.json` (written by
`scripts/boss_mock.py`).

In [ ]:
# data/rrab.parquet: all RRab within 20 deg, with the flags ogle_own, in_gaia and the OGLE mean I magnitude
t = pd.read_parquet(DATA / "rrab.parquet")
# results/boss_mock.json: recovered / true weighted counts per true [Fe/H] bin over the mock realisations
mock = json.loads((ROOT / "results" / "boss_mock.json").read_text())

#### The RR Lyrae
OGLE-own RRab and *Gaia* RRab inside the OGLE-IV footprint, in the orthographic projection about the LMC centre.

In [ ]:
t = t[t.rho < 20].copy()   # rho [deg] from the LMC centre
t["in_fp"] = ogle.in_footprint(t.ra.to_numpy(), t.dec.to_numpy())   # inside the OGLE-IV fields
# Orthographic x (east), y (north) [deg] about the LMC centre.
t["x"], t["y"] = lmc_xy(t.ra.to_numpy(), t.dec.to_numpy())
og, ga = t[t.ogle_own], t[t.in_gaia & t.in_fp]  # OGLE-own RRab; Gaia RRab inside the OGLE-IV footprint


def binned(s, flag, col, edges):
    """Fraction of s with `flag` in bins of `col`, binomial error; NaN where fewer than 20 stars.

    Parameters
    ----------
    s : pandas.DataFrame
        Stars.
    flag : str
        Boolean column whose mean is the fraction.
    col : str
        Binned column (here mag [mag]).
    edges : numpy.ndarray
        Bin edges in `col`.

    Returns
    -------
    centres, fraction, error : numpy.ndarray
        Bin centres, fraction with `flag` and sqrt(f (1 - f) / n) per bin.
    """
    k = np.digitize(s[col], edges) - 1   # bin of each star
    # Stars per bin, and the fraction with `flag` in bins of at least 20 stars.
    n = np.array([(k == i).sum() for i in range(len(edges) - 1)])
    f = np.array([s[flag][k == i].mean() if n[i] >= 20 else np.nan for i in range(len(edges) - 1)])
    return 0.5 * (edges[1:] + edges[:-1]), f, np.sqrt(f * (1 - f) / np.maximum(n, 1))

#### Panel (a): *Gaia* completeness against magnitude
In three rings of $\rho$; OGLE $I + 0.6$ stands in for *Gaia* $G$.

In [ ]:
fig, ax = panel_grid(1, 4, hgap=0.55, top=0.7)   # four equal panels in a row
a = ax[0, 0]
m = og.assign(mag=og.I + 0.6)  # OGLE I + 0.6 ~ Gaia G for LMC RRab
# Rings in rho [deg]; 0.25-mag bins from 17.75 mag.
for (lo, hi), col in zip(((0, 2), (2, 4), (4, 12)), (BLUE, ORANGE, GREEN)):
    x, f, e = binned(m[(m.rho >= lo) & (m.rho < hi)], "in_gaia", "mag", np.arange(17.75, 20.6, 0.25))
    a.errorbar(x, f, yerr=e, fmt="o-", ms=2.5, lw=1, color=col, label=f"{lo} < $\\rho$ < {hi} deg")
a.set(xlim=(17.8, 20.6), ylim=(0, 1.04), xlabel=r"$I + 0.6 \approx G$ [mag]", ylabel="Gaia completeness")
a.legend(loc="lower left", bbox_to_anchor=(0.0, 1.02), ncol=1, borderaxespad=0.1, handlelength=1.5)
a.text(0.95, 0.05, "(a)", transform=a.transAxes, ha="right")

#### Panels (b, c): completeness on the sky
Counts per 1-deg pixel (written to `results/rrab_selection.csv`) and the outline of the OGLE-IV footprint.

In [ ]:
# Completeness per 1-deg pixel of the orthographic x, y inside the OGLE-IV footprint (pixels with >= 5 stars found by both).
e = np.arange(-12, 12.01, 1.0)   # deg, pixel edges in x and y
rows = []   # one dict per pixel kept
for i in range(len(e) - 1):
    for j in range(len(e) - 1):
        # Stars of pixel (i, j) inside the footprint; OGLE-own, Gaia and both.
        s = t[t.in_fp & (t.x >= e[i]) & (t.x < e[i + 1]) & (t.y >= e[j]) & (t.y < e[j + 1])]
        nO, nG, nB = int(s.ogle_own.sum()), int(s.in_gaia.sum()), int((s.ogle_own & s.in_gaia).sum())
        if nB >= 5:
            rows.append(dict(ix=i, iy=j, x=0.5 * (e[i] + e[i + 1]), y=0.5 * (e[j] + e[j + 1]), n_ogle=nO, n_gaia=nG, n_both=nB,
                             C_gaia=nB / nO, C_ogle=nB / nG, C_union=(nO + nG - nB) / (nO * nG / nB)))
p = pd.DataFrame(rows)   # ix, iy: pixel indices for the maps; dropped from the csv
p.drop(columns=["ix", "iy"]).to_csv(ROOT / "results" / "rrab_selection.csv", index=False, float_format="%.4f")
# Footprint outline: a 400 x 400 grid of (x, y) mapped back to (ra, dec) with rho = hypot(x, y) and PA = atan2(x, y). This is
# the inverse of an azimuthal equidistant projection; lmc_xy is orthographic (x = sin(rho) sin(PA)), so near rho = 12.5 deg the
# outline lies about 0.1 deg farther from the centre than the stars it encloses (sin(12.6 deg) = 12.5 deg in radians).
g = np.linspace(-L, L, 400)   # deg, grid in x and y
gx, gy = np.meshgrid(g, g)
r0 = np.deg2rad(np.hypot(gx, gy))   # rad, angular distance from the centre
pa = np.arctan2(gx, gy)   # rad, position angle (north through east)
# Spherical trigonometry: (ra, dec) at distance r0 and position angle pa from the LMC centre (lmcold.constants).
dec0, ra0 = np.deg2rad(LMC["dec0"]), np.deg2rad(LMC["ra0"])
dec = np.arcsin(np.sin(dec0) * np.cos(r0) + np.cos(dec0) * np.sin(r0) * np.cos(pa))
ra = ra0 + np.arctan2(np.sin(pa) * np.sin(r0) * np.cos(dec0), np.cos(r0) - np.sin(dec0) * np.sin(dec))
# (400, 400) bool map: grid point inside the OGLE-IV footprint.
fp = ogle.in_footprint(np.rad2deg(ra).ravel(), np.rad2deg(dec).ravel()).reshape(gx.shape)
fp = binary_fill_holes(binary_closing(fp, np.ones((5, 5))))  # close the ~0.1 deg gaps between field rectangles (grid step 0.06 deg)
CMAP = ListedColormap(plt.cm.Blues(np.linspace(0.95, 0.15, 256)))  # completeness 1 is light blue, not white
# (b) and (c): map of the pixel completeness (NaN: pixel not kept) and the footprint outline.
for a, col, lab in ((ax[0, 1], "C_gaia", "(b) Gaia"), (ax[0, 2], "C_ogle", "(c) OGLE, own search")):
    mm = np.full((len(e) - 1, len(e) - 1), np.nan)
    mm[p.ix, p.iy] = p[col]
    im = a.pcolormesh(e, e, mm.T, cmap=CMAP, vmin=0.4, vmax=1.0, rasterized=True)
    a.contour(g, g, fp.astype(float), levels=[0.5], colors="0.3", linewidths=0.6)
    a.set(xlim=(L, -L), ylim=(-L, L), aspect="equal", xticks=[10, 5, 0, -5, -10], yticks=[-10, -5, 0, 5, 10],
          xlabel=r"$x$ (east) [deg]")
    a.text(0.0, 1.02, lab, transform=a.transAxes, fontsize=7)
ax[0, 1].set_ylabel(r"$y$ (north) [deg]")
ax[0, 2].tick_params(labelleft=False)
# One colour bar spanning panels (b) and (c): two axes widths plus the gap (hgap 0.55 inch) in units of the axes width.
cax = ax[0, 1].inset_axes([0.0, 1.16, 2.0 + 0.55 / (ax[0, 1].get_position().width * fig.get_size_inches()[0]), 0.05])
fig.colorbar(im, cax=cax, orientation="horizontal")
cax.xaxis.set_ticks_position("top")
cax.tick_params(labelsize=6)
cax.set_title("completeness per 1 deg pixel; grey line: OGLE-IV footprint", fontsize=7, pad=9)

#### Panel (d): the BOSS mock
Recovered over true counts per bin of true [Fe/H] (`results/boss_mock.json`, written by `scripts/boss_mock.py`).

In [ ]:
a = ax[0, 3]
me = np.array(mock["feh_edges"])   # dex, true [Fe/H] bin edges of the mock
# Selection-group weights (grey) and final weights (blue), offset by -+0.02 dex; mean and sd over the realisations.
for key, lab, col, dx in (("group", "selection-group weights", "0.55", -0.02), ("final", "final weights", BLUE, 0.02)):
    q = mock["obs_bias_1.0"][key]["binned"]
    a.errorbar(0.5 * (me[1:] + me[:-1]) + dx, q["mean"], yerr=q["sd"], fmt="o-", ms=3, lw=1, color=col, label=lab)
a.axhline(1, color="0.6", lw=0.6)
a.set(xlabel="true [Fe/H]", ylabel="recovered / true", xlim=(-3.1, 0.6), ylim=(0.5, 1.7))
a.legend(loc="lower left", bbox_to_anchor=(0.0, 1.02), ncol=1, borderaxespad=0.1, handlelength=1.5, title="BOSS mock", alignment="left")
a.text(0.95, 0.05, "(d)", transform=a.transAxes, ha="right")
fig.savefig(ROOT / "figures" / "paper_selection_checks.png")
# Median completeness over the kept pixels.
print(p[["C_gaia", "C_ogle", "C_union"]].median().round(3).to_dict())

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_selection_checks.png')

**Figure 2.2.** Checks of the selection functions. (a) *Gaia* completeness of OGLE's own RRab against
$I+0.6\approx G$ (mag; OGLE mean $I$ magnitude), in 0.25-mag bins with at least 20 stars, in three rings of $\rho$: 0–2° (blue),
2–4° (orange) and 4–12° (green); binomial errors. (b, c) *Gaia* (b) and OGLE own-search (c) completeness in 1° pixels of the
orthographic projection, $x$ (east, increasing to the left) and $y$ (north) in degrees, for pixels with at least 5 stars found by
both (colour scale 0.4–1, light = complete); grey line: OGLE-IV footprint. (d) BOSS mock on the real design: weighted counts over
true counts per bin of true [Fe/H] (dex), with the selection-group weights alone (grey) and the final weights (blue); mean and
standard deviation of 50 realisations. `scripts/paper_selection_checks.py`, `scripts/boss_mock.py`

#### Running the BOSS mock

`scripts/boss_mock.py` runs the mock on the real BOSS design: the weight test with and without the observation bias, and the
coverage test with the two failure models. It writes `results/boss_mock.json`, from which panel (d) of
Figure 2.2 and the numbers above are taken.

<details><summary>About <code>scripts/boss_mock.py</code></summary>

```text
Mock of the BOSS selection on the real design, with known true [Fe/H], to test the weights of lmcold.selection.

Real: eligible parent, selection groups, observed flags, G, which stars have XP [M/H] (data/boss_weights.parquet).
Mock:
  true [Fe/H]: per eligible star, the BOSSNet [Fe/H] of a good spectrum from the same nside-16 pixel drawn with probability
    proportional to its weight, de-compressed about FEH_PIV by 1 / COMP, plus N(0, 0.05);
  pass: P = p_s T, T = 1 - A(G) clip((-1 - [Fe/H]) / 1.5, 0, 1)^1.5, A(G) = 0.7 clip((G - 16.25) / 1.25, 0, 1) (shaped like the
    measured deficit, not like the estimator); p_s keeps the real pass rate of the selection group (capped so P <= 1);
  XP [M/H]: true [Fe/H] + N(0, XP_SD) where the real star has XP [M/H] (XP_SD = robust scatter of Andrae et al. 2023 about
    APOGEE DR17 for the LMC giants);
  variant obs_bias: observation itself favours [Fe/H] < -1.5 by a factor 1.3 within each selection group (keeping its N_obs),
    which violates the assumption of the weights.
For NREAL realisations per variant: weighted count of good spectra / true count of covered eligible stars, in bins of true
[Fe/H] (FEH_EDGES) and below -1.5 and -2, for the group weights alone and the final weights (group weights x pass factor,
lmcold.selection, as scripts/boss_selection.py).
Fill test (NFILL realisations per failure model; failure 'central' doubles A(G) within 3 deg, as the real spectra fail more in the
crowded centre): final weights x coverage factor (lmcold.selection.coverage_factor) over the true count of ALL eligible stars,
below -1.5 and -2 and in total, per ring (0-3, 3-6, 6-12 deg). Fills: 'ring' (keys ring x G, G; the current one) and 'xp'
(region x G x XP bin, region x G, ring x G, G; regions = 4 rings x quadrants, FILL_NMIN covered stars per cell). Coverage:
'standard' (>= NMIN_GOOD good spectra) and 'sparse_gap' (also good / parent >= SPARSE: sparse groups filled like gaps).
Writes results/boss_mock.json.

Input: data/boss_weights.parquet (release_boss_weights, scripts/boss_selection.py). The mock pass model T is floored at 0.05.
results/boss_mock.json: feh_edges; obs_bias_1.0 and obs_bias_1.3, each with group/final x binned (per FEH_EDGES bin) and
cumulative (below -1.5, -2) mean and sd over NREAL realisations of recovered / true; fill: real and central failure models,
each with '<coverage>/<fill>' -> ring0..ring2 -> [ratio below -1.5, below -2, all stars] mean and sd over NFILL realisations.
The mock serves the BOSS truth-mock panel of figures/paper_selection_checks.png (appendix, Sect. 2.2).

Run: .venv/bin/python scripts/boss_mock.py   (after boss_selection.py)
```
</details>

*Shown, not run here: data/boss_weights.parquet (scripts/boss_selection.py); 140 mock realisations. The cells below are the script itself; run it from the repository with* `python scripts/boss_mock.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import json

import numpy as np
import pandas as pd
import healpy as hp

from lmcold import selection
from lmcold.constants import LMC
from lmcold.footprint import angular_distance, lmc_xy
from lmcold.groups import bin_index
from lmcold.sources import DATA, ROOT
```

##### Settings
Numbers of realisations and the parameters of the mock.

```python
NREAL = 50  # realisations per observation-bias variant
NFILL = 20  # realisations per failure model in the fill test
FILL_NMIN = 20  # covered stars needed in a fill cell (lmcold.selection.coverage_factor)
SPARSE = 0.01  # 'sparse_gap' coverage: groups with good / parent < 1% are treated as gaps
FILL_G_EDGES = [12.0, 15.5, 16.25, 16.75, 17.1, 17.5]  # G bins [mag] of the fill cells
FEH_PIV, COMP = -0.6, 0.85  # [dex], -: BOSSNet [Fe/H] is stretched about FEH_PIV by 1 / COMP to make the true [Fe/H]
XP_SD = 0.11  # dex; Andrae+23 XP [M/H] scatter about APOGEE for LMC giants (correlation 0.85)
CUTS = (-1.5, -2.0)  # [Fe/H] thresholds [dex] of the cumulative ratios
FEH_EDGES = [-3.0, -2.25, -2.0, -1.75, -1.5, -1.25, -1.0, -0.5, 0.5]  # true [Fe/H] bins [dex]
```

##### Load the data
The BOSS weights `data/boss_weights.parquet` (`scripts/boss_selection.py`): the real parent,
selection groups, observed flags and the stars with XP [M/H].

```python
# The real design: one row per eligible parent star.
c = pd.read_parquet(DATA / "boss_weights.parquet")
```

##### The real design
Group index, real pass rate, donors of true [Fe/H], rings and fill regions of each star.

```python
# s: integer selection-group index per star; g: Gaia G [mag]; real observed and good-spectrum flags
s = pd.factorize(c.group)[0]
g, obs_real, good_real = c.g.to_numpy(), c.observed.to_numpy(), c.good.to_numpy()
# Stars with XP [M/H]; pix: their HEALPix nside-16 nested pixel (donor pools of true_feh)
has_xp = c.mh_xp.notna().to_numpy()
pix = hp.ang2pix(16, np.radians(90 - c.dec.to_numpy()), np.radians(c.ra.to_numpy()), nest=True)
# Real pass rate N_good / N_obs of each star's selection group.
p_real = (np.bincount(s, good_real) / np.maximum(np.bincount(s, obs_real), 1))[s]
# Donors of true [Fe/H]: good spectra with positive weight, labelled by nside-16 pixel.
donors = c[c.good & (c.w > 0)].assign(pix=pix[(c.good & (c.w > 0)).to_numpy()])
# rho: angular distance [deg] from the adopted LMC centre; ring: 3-ring index of the fill test
rho = angular_distance(c.ra.to_numpy(), c.dec.to_numpy(), LMC["ra0"], LMC["dec0"])
ring = np.digitize(rho, [3, 6])  # 0, 1, 2 for 0-3, 3-6, >= 6 deg
# Projected coordinates [deg] (x east, y north) about the LMC centre
x, y = lmc_xy(c.ra.to_numpy(), c.dec.to_numpy())
# Fill region: 4 rings (0-3, 3-6, 6-9, >= 9 deg) x 4 quadrants of the projected (x east, y north) plane.
region = 4 * np.clip(np.digitize(rho, [3, 6, 9]), 0, 3) + (x >= 0) + 2 * (y >= 0)
# G bin of each star for the fill cells
gfill = bin_index(g, FILL_G_EDGES)
n_parent = np.bincount(s)[s]  # eligible stars in each star's selection group
```

##### Mock ingredients
`true_feh` draws a true [Fe/H] for every star; `observe` returns the observed flags, optionally
biased towards metal-poor stars.

```python
def true_feh(rng):
    """Draw a true [Fe/H] for every eligible star.

    Parameters
    ----------
    rng : numpy.random.Generator

    Returns
    -------
    numpy.ndarray
        True [Fe/H] [dex] per star: BOSSNet [Fe/H] of a weight-drawn donor in the same nside-16 pixel (any donor if the pixel
        has none), stretched about FEH_PIV by 1 / COMP, plus N(0, 0.05 dex).
    """
    out = np.full(len(c), np.nan)
    # For each pixel with donors: draw its stars' values from those donors, probability proportional to weight w
    for p, d in donors.groupby("pix"):
        k = np.flatnonzero(pix == p)
        out[k] = rng.choice(d.feh.to_numpy(), len(k), p=(d.w / d.w.sum()).to_numpy())
    # Stars in pixels without donors draw from all donors
    left = np.isnan(out)
    out[left] = rng.choice(donors.feh.to_numpy(), left.sum(), p=(donors.w / donors.w.sum()).to_numpy())
    # Stretch about FEH_PIV by 1 / COMP and add N(0, 0.05 dex) scatter
    return FEH_PIV + (out - FEH_PIV) / COMP + rng.normal(0, 0.05, len(c))


def observe(f, rng, bias):
    """Real observed flags, or the same number per selection group drawn with weight `bias` for [Fe/H] < -1.5.

    Parameters
    ----------
    f : numpy.ndarray
        True [Fe/H] per star [dex].
    rng : numpy.random.Generator
    bias : float
        Relative probability of observing a star with [Fe/H] < -1.5; 1 returns the real flags.

    Returns
    -------
    numpy.ndarray of bool
        Observed flag per star; N_obs per selection group equals the real one.
    """
    if bias == 1:
        return obs_real
    key = np.log(rng.random(len(c))) / np.where(f < -1.5, bias, 1.0)  # Efraimidis-Spirakis weighted sampling keys
    # Within each group, the N_obs stars with the largest keys are observed.
    rank = pd.Series(-key).groupby(s).rank(method="first").to_numpy()
    return rank <= np.bincount(s, obs_real)[s]
```

##### One realisation
Failures that depend on $G$ and true [Fe/H], mock XP [M/H], the weights, and the recovered over
true counts.

```python
def realisation(seed, bias, central=False, fill=False):
    """One mock realisation of the BOSS selection and the recovered / true ratios.

    Parameters
    ----------
    seed : int
        Random seed.
    bias : float
        Observation bias for [Fe/H] < -1.5 (see observe).
    central : bool
        Double the failure amplitude A(G) within 3 deg of the centre.
    fill : bool
        Run the fill (coverage) test instead of the weight test.

    Returns
    -------
    dict
        fill=False: {'group'|'final': {'binned': ratio per FEH_EDGES bin, 'cumulative': ratio below each of CUTS}}, with the
        truth = covered eligible stars. fill=True: {'<coverage>/<fill>': {'ring<r>': [ratio < -1.5, ratio < -2, ratio all]}},
        with the truth = all eligible stars in the ring.
    """
    rng = np.random.default_rng(seed)
    # True [Fe/H] [dex] and observed flags of this realisation
    f = true_feh(rng)
    obs = observe(f, rng, bias)
    # Pass model: amplitude A(G) rises from 0 at G = 16.25 to 0.7 at G >= 17.5; T falls with [Fe/H] below -1.
    a = 0.7 * np.clip((g - 16.25) / 1.25, 0, 1) * np.where(central & (rho < 3), 2.0, 1.0)
    t = np.clip(1 - a * np.clip((-1 - f) / 1.5, 0, 1) ** 1.5, 0.05, 1)
    # Normalise T within each group so the expected pass rate of the observed stars equals the real one.
    mean_t = np.bincount(s, obs * t) / np.maximum(np.bincount(s, obs), 1)
    # Good spectrum: observed and passing with probability P = p_s T, capped at 1
    good = obs & (rng.random(len(c)) < np.minimum(1, p_real / np.maximum(mean_t[s], 1e-9) * t))
    # Mock XP [M/H] only where the real star has one.
    mh = np.where(has_xp, f + rng.normal(0, XP_SD, len(c)), np.nan)
    # Weights as in scripts/boss_selection.py: w0 group weights alone; w = w0 x pass factor (final weights)
    ones = np.ones(len(c))
    w0 = selection.group_weights(s, good, ones)
    w = w0 * selection.pass_factor(s, obs, good, selection.boss_pass_groups(g, mh, c.ra, c.dec), ones)
    # Good spectra in each star's group; covered: groups with >= NMIN_GOOD good spectra
    n_good = np.bincount(s, good)[s]
    covered = n_good >= selection.NMIN_GOOD
    if fill:
        # Fill keys, finest first: integer codes of (ring, G) or (region, G, XP bin); stars without XP get their own bin.
        xpbin = np.where(np.isfinite(mh), bin_index(mh, selection.XP_EDGES), len(selection.XP_EDGES))
        keys = dict(ring=[ring * 10 + gfill, gfill], xp=[(region * 10 + gfill) * 10 + xpbin, region * 10 + gfill, ring * 10 + gfill, gfill])
        # For each coverage rule and fill: final weights x coverage factor, summed per ring below each cut and in total,
        # over the true count of all eligible stars there
        out = {}
        for cov_name, cov in (("standard", covered), ("sparse_gap", covered & (n_good / n_parent >= SPARSE))):
            for fill_name, k in keys.items():
                ww = w * cov * selection.coverage_factor(k, cov, FILL_NMIN, ones)
                out[f"{cov_name}/{fill_name}"] = {f"ring{r}": [float(np.sum(ww[(ring == r) & (f < x)]) / np.sum((ring == r) & (f < x)))
                                                               for x in CUTS] + [float(np.sum(ww[ring == r]) / np.sum(ring == r))]
                                                  for r in range(3)}
        return out
    k = np.digitize(f, FEH_EDGES)  # 0 below the first edge; bins 1 .. len - 1
    # True number of covered eligible stars per FEH_EDGES bin
    true = np.bincount(k[covered], minlength=len(FEH_EDGES) + 1)[1:len(FEH_EDGES)]
    out = {}
    # Recovered / true per bin and below each of CUTS, for the group and the final weights
    for lab, ww in (("group", w0), ("final", w)):
        out[lab] = dict(binned=(np.bincount(k, ww, len(FEH_EDGES) + 1)[1:len(FEH_EDGES)] / true).tolist(),
                        cumulative=[float(np.sum(ww[f < x]) / np.sum(covered & (f < x))) for x in CUTS])
    return out
```

##### Weight test
Realisations without and with an observation bias.

```python
# NREAL realisations (seeds 0 .. NREAL - 1) without (1.0) and with (1.3) observation bias; mean and sd to 3 decimals
res = dict(feh_edges=FEH_EDGES)
for bias in (1.0, 1.3):
    r = [realisation(seed, bias) for seed in range(NREAL)]
    res[f"obs_bias_{bias}"] = {lab: {kind: dict(mean=np.round(np.mean([q[lab][kind] for q in r], 0), 3).tolist(),
                                                sd=np.round(np.std([q[lab][kind] for q in r], 0), 3).tolist())
                                     for kind in ("binned", "cumulative")} for lab in ("group", "final")}
```

##### Coverage test
Realisations of the fill test for the real and the central failure models.

```python
# Column names of the fill ratios; NFILL realisations (seeds from 1000) for each failure model, mean and sd
res["fill"] = dict(columns=[f"<{x}" for x in CUTS] + ["all"])
for central in (False, True):
    r = [realisation(1000 + seed, 1.0, central, fill=True) for seed in range(NFILL)]
    res["fill"]["central" if central else "real"] = {v: {ring_: dict(mean=np.round(np.mean([q[v][ring_] for q in r], 0), 3).tolist(),
                                                                    sd=np.round(np.std([q[v][ring_] for q in r], 0), 3).tolist())
                                                         for ring_ in r[0][v]} for v in r[0]}
```

##### Write the results
`results/boss_mock.json`.

```python
# Write results/boss_mock.json and print it
(ROOT / "results" / "boss_mock.json").write_text(json.dumps(res, indent=1))
print(json.dumps(res, indent=1))
```

### Coverage gaps and non-members

Figure 2.3 shows where BOSS sampled the parent and checks assumptions A4 and A5 on the data. The fraction of the
parent with a good spectrum is high over most of the disc and lowest in the bar region and at the outer edge; the uncovered groups
lie mostly in the south-east. The parent stars of uncovered groups have the XP metallicities of the covered stars in the same rings
and $G$ bins: the fractions below [M/H]$_{\rm XP}$ = -1, -1.5 and -1.75 are 0.176,
0.0169 and 0.0043 against 0.179, 0.0177 and
0.0055 (bootstrap over uncovered groups: ±0.017, ±0.0019 and
±0.0006). The deficit below -1.75 means the coverage factor puts slightly too many
metal-poor stars into the gaps, about 1.5 per cent of the most metal-poor bin. Non-members are
0.0–2.3 per cent of the weighted good BOSS spectra, rising towards faint $G$, and
at most 0.4 per cent of the APOGEE ones.

<details><summary>About <code>scripts/paper_selection_gaps.py</code></summary>

```text
Paper figure (appendix, Sect. 2.2): BOSS coverage gaps and non-members in the parent.

Uncovered parent stars are those in selection groups with fewer than selection.NMIN_GOOD good spectra (data/boss_weights.parquet).
(a) Fraction of the parent with a good spectrum per HEALPix nside=64 pixel; red outlines: pixels with most of their parent in
uncovered groups. (b) Fraction of XP [M/H] below a threshold, uncovered
parent over covered parent reweighted to the same ring (0-3-6-12 deg) x G (edges 12, 15.5, 16.25, 16.75, 17.1, 17.5) mix, i.e. the
assumption of the coverage factor; band: 16th-84th percentiles of a bootstrap over uncovered groups. (c) Weighted fraction of good
spectra that are not LMC giants (Teff >= 5500 K or log g >= 3.5, or v_los outside 150-420 km/s), BOSS per ring x G and APOGEE per
field type. Also (json only): the fraction of APOGEE velocity members (good-spectrum giants, 150 < v < 420 km/s; APOGEE targeted
without astrometry) that fail the carton's astrometric cuts, by field type; and the sensitivity of the weighted BOSS member counts
(final weights x ring x G coverage factor) below [Fe/H] = -1.5, -2 and in total to the coverage threshold NMIN_GOOD = 2, 3, 5.
Writes figures/paper_selection_gaps.png and results/selection_gaps.json.

Inputs: data/boss_weights.parquet (release_boss_weights), data/apogee_weights.parquet (release_apogee_weights),
data/apogee_parent.parquet (release_apogee_parent; APOGEE candidates with nbr = bright-neighbour flag, astro_ok = passes the
carton's astrometric cuts, spectroscopic columns), data/spec_giants.parquet (scripts/build_spec_giants.py; BOSS and APOGEE
v [km/s], teff [K], logg), lmcold.mass_samples.samples() (BOSS members, for the coverage-threshold test).
Note: (c) uses the raw BOSS velocity v of data/spec_giants.parquet, not the calibrated v_los of the mass samples, and keeps the
first row per Gaia source_id.
Panels (1 x 3, square):
(a) x (east, increasing to the left), y (north) [deg], +-12.5, orthographic about the LMC centre; nside-64 pixels coloured by
    good spectra / parent (log, 1e-3 to 1); red outlines: pixels where more than half of the parent is in uncovered groups.
(b) x: XP [M/H] threshold, -2 to -0.5 in 0.125 steps; y: fraction below the threshold (log); orange: uncovered parent with
    16th-84th percentile band from 300 bootstraps over uncovered groups; blue: covered parent reweighted to the ring x G mix of
    the uncovered stars.
(c) x: G bin (labels <15.5, 15.5-16.25, 16.25-16.75, 16.75-17.1, 17.1-17.5); y: weighted per cent of good spectra that are not
    LMC giants, 0-7; BOSS per ring (0-3, 3-6, 6-12 deg); horizontal lines: APOGEE LMC1-17 (dashed) and LMC18-35 (dotted).
JSON keys: parent, uncovered, uncovered_share, uncovered_by_ring, xp_available, xp_gaps, boss_nonmember, apogee_nonmember,
sparse_groups, coverage_factor (parent / covered per ring x G bin and per ring), parent_xp_fraction_covered,
astrometric_loss_of_members, coverage_threshold (weighted member counts relative to NMIN_GOOD = 3).
Placement: appendix (Sect. 2.2; docs/figures.md). Run: .venv/bin/python scripts/paper_selection_gaps.py   (about 4 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_selection_gaps.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_parent.parquet` | Zenodo bundle 1 | APOGEE-2 LMC1-35 RGB-polygon candidates (field x star) with Gaia DR3 (2MASS best neighbour), observed flags and allStarLite columns |
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import json

import healpy as hp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd

from lmcold import giants, selection
from lmcold.groups import bin_index
from lmcold.mass_samples import samples
from lmcold.constants import LMC
from lmcold.footprint import angular_distance, lmc_xy
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
BLUE, ORANGE = '#2878b5', '#d75a20'  # line colours: covered / BOSS, uncovered / outer ring
G_EDGES = [15.5, 16.25, 16.75, 17.1]  # G bin edges [mag] (inner edges of 12-15.5-16.25-16.75-17.1-17.5)
THRESHOLDS = np.arange(-2.0, -0.49, .125)  # XP [M/H] thresholds [dex] of panel (b): -2 to -0.5 in 0.125 steps
rng = np.random.default_rng(5)  # fixed seed of the bootstrap over uncovered groups

#### Load the data
The selection weights of the two samples (`data/boss_weights.parquet`, `data/apogee_weights.parquet`), the APOGEE targeting
candidates (`data/apogee_parent.parquet`) and the spectroscopic parameters (`data/spec_giants.parquet`), all from Zenodo
bundle 1; and the BOSS members of the mass samples through `lmcold.mass_samples.samples()`.

In [ ]:
# data/boss_weights.parquet: the BOSS parent with selection group, good/observed flags, weight w, G and XP [M/H]
b = pd.read_parquet(DATA / 'boss_weights.parquet')
# data/spec_giants.parquet: BOSS and APOGEE spectra with raw velocity v [km/s], teff [K], logg
v = pd.read_parquet(DATA / 'spec_giants.parquet')
# data/apogee_weights.parquet: the APOGEE candidates with field, quality flag and weight w
a = pd.read_parquet(DATA / 'apogee_weights.parquet')
# data/apogee_parent.parquet: APOGEE candidates with bright-neighbour flag nbr, astrometric-cut flag astro_ok, spectra; one row per
# star (observed first)
ap = pd.read_parquet(DATA / 'apogee_parent.parquet').sort_values(['apogee_id', 'observed'], ascending=[True, False]).drop_duplicates('apogee_id')
# lmcold.mass_samples.samples(): the BOSS members (calibrated [Fe/H]) for the coverage-threshold test
members, _, _ = samples()

#### The BOSS parent
Covered and uncovered selection groups, rings and $G$ bins of the parent stars.

In [ ]:
# Covered: the star's selection group holds at least NMIN_GOOD good spectra.
b['covered'] = b.group.map(b.groupby('group').good.sum()).to_numpy() >= selection.NMIN_GOOD
# Projected distance [deg] from the adopted LMC centre.
b['rho'] = angular_distance(b.ra.to_numpy(), b.dec.to_numpy(), LMC['ra0'], LMC['dec0'])
b['ring'] = np.digitize(b.rho, [3, 6])  # 0: < 3 deg, 1: 3-6 deg, 2: >= 6 deg
b['gb'] = np.digitize(b.g, G_EDGES)  # G bin 0-4
# Counts of the parent and of the uncovered stars, overall and per ring; share with an XP [M/H] among covered and uncovered.
out = dict(parent=len(b), uncovered=int((~b.covered).sum()), uncovered_share=float((~b.covered).mean()),
           uncovered_by_ring={int(k): int(v) for k, v in (~b.covered).groupby(b.ring).sum().items()},
           xp_available=dict(covered=float(np.isfinite(b.mh_xp[b.covered]).mean()), uncovered=float(np.isfinite(b.mh_xp[~b.covered]).mean())))

#### XP metallicities of the gaps
Uncovered parent stars against covered ones reweighted to the same ring and $G$ mix, with a bootstrap over uncovered groups.

In [ ]:
# (b) XP metallicities of the gaps against the covered stars of the same ring x G mix.
x = b[np.isfinite(b.mh_xp)]  # parent stars with an XP [M/H]
u, c = x[~x.covered], x[x.covered]  # uncovered and covered
mix = u.groupby(['ring', 'gb']).size() / len(u)
# Weight of each covered star: share of its ring x G cell among the uncovered stars over its share among the covered stars.
wc = (mix / (c.groupby(['ring', 'gb']).size() / len(c))).reindex(pd.MultiIndex.from_frame(c[['ring', 'gb']])).fillna(0).to_numpy()
frac = lambda mh, w: np.array([np.average(mh < t, weights=w) for t in THRESHOLDS])  # noqa: E731  fraction below each threshold
# Fractions below each threshold: covered (reweighted) and uncovered (unweighted).
fc, fu = frac(c.mh_xp.to_numpy(), wc), frac(u.mh_xp.to_numpy(), None)
groups = u.group.unique()
boot = []
# 300 bootstrap draws of the uncovered groups (with replacement); a group drawn n times enters with weight n.
for _ in range(300):
    n = pd.Series(rng.choice(groups, len(groups))).value_counts()  # times each uncovered group is drawn
    s = u[u.group.isin(n.index)]
    boot.append(frac(s.mh_xp.to_numpy(), s.group.map(n).to_numpy()))
lo, hi = np.percentile(np.array(boot), [16, 84], axis=0)  # band of panel (b)
out['xp_gaps'] = [dict(below=float(t), uncovered=float(a), uncovered_p16=float(l), uncovered_p84=float(h), covered_same_mix=float(z))
                  for t, a, l, h, z in zip(THRESHOLDS, fu, lo, hi, fc)]

#### Non-members, sparse groups, coverage factors and astrometric losses
The numbers written to `results/selection_gaps.json`.

In [ ]:
# (c) Non-members among the weighted good spectra.
# First BOSS row per star (raw velocity, Teff, log g), joined to the good spectra with a weight.
vb = v[v.survey == 'boss'][['source_id', 'v', 'teff', 'logg']].drop_duplicates('source_id')
g = b[b.good.astype(bool) & (b.w > 0)].merge(vb, on='source_id', how='left')
# LMC giant: the giant cuts of lmcold.giants.is_giant and the 150-420 km/s window of lmcold.mass_samples; NaN fails.
g['member'] = (g.teff < 5500) & (g.logg < 3.5) & (g.v > 150) & (g.v < 420)
# Weighted non-member fraction per ring x G bin (panel c), overall and per ring.
boss_nm = g.groupby(['ring', 'gb']).apply(lambda s: 1 - np.average(s.member, weights=s.w), include_groups=False)
out['boss_nonmember'] = dict(all=float(1 - np.average(g.member, weights=g.w)),
                             by_ring={int(k): float(1 - np.average(s.member, weights=s.w)) for k, s in g.groupby('ring')})
# APOGEE: the same per field type, for weighted good-quality spectra (velocity v from the weights table).
va = v[v.survey == 'apogee'][['source_id', 'teff', 'logg']].drop_duplicates('source_id')
a = a[a.quality.astype(bool) & (a.w > 0)].merge(va, on='source_id', how='left')
a['member'] = (a.teff < 5500) & (a.logg < 3.5) & (a.v > 150) & (a.v < 420)
a['type'] = np.where(a.field.str[3:].astype(int) <= 17, 'LMC1-17', 'LMC18-35')  # field number from 'LMCn'
out['apogee_nonmember'] = {k: dict(n=len(s), unweighted=float(1 - s.member.mean()), weighted=float(1 - np.average(s.member, weights=s.w)))
                           for k, s in a.groupby('type')}
# Sparsely sampled covered groups (the holes of the sky maps): good / parent < 1 per cent.
gs = b.groupby('group').agg(parent=('good', 'size'), good=('good', 'sum'))
sparse = gs[(gs.good >= selection.NMIN_GOOD) & (gs.good / gs.parent < 0.01)]
inside = b.group.isin(sparse.index)
out['sparse_groups'] = dict(groups=len(sparse), parent_share=float(sparse.parent.sum() / len(b)), good=int(sparse.good.sum()),
                            median_rho=float(b.rho[inside].median()), max_w=float(b.w.max()))
# Coverage factor: parent over covered parent per ring x G bin and per ring.
out['coverage_factor'] = {f'ring{r}': np.round(b[b.ring == r].groupby('gb').size() / b[(b.ring == r) & b.covered].groupby('gb').size(), 3).tolist()
                          for r in range(3)} | {'per_ring': np.round(b.groupby('ring').size() / b[b.covered].groupby('ring').size(), 3).tolist()}
# Fraction of the covered parent with XP [M/H] below -1, -1.5 and -2.
out['parent_xp_fraction_covered'] = {f'<{x}': float(np.mean(b.mh_xp[b.covered & np.isfinite(b.mh_xp)] < x)) for x in (-1.0, -1.5, -2.0)}

# Members lost to the astrometric cuts, from the fields APOGEE targeted without astrometry.
ap = ap[~ap.nbr & ap.observed & ap.snr.notna()]  # one row per star (observed first), no bright neighbour, with a spectrum
# Good spectrum at S/N >= 30, giant cuts, 150 < v < 420 km/s.
ap = ap[giants.apogee_spectrum_ok(ap, 30) & giants.is_giant(ap) & (ap.v > 150) & (ap.v < 420)]
ap['type'] = np.where(ap.field.str[3:].astype(int) <= 17, 'LMC1-17', 'LMC18-35')
out['astrometric_loss_of_members'] = {k: dict(n=len(t), fail=float(1 - t.astro_ok.mean())) for k, t in ap.groupby('type')}

#### Coverage threshold
Weighted member counts with two, three or five good spectra required per covered group.

In [ ]:
# Coverage threshold: members (lmcold.mass_samples) weighted with NMIN_GOOD = 2, 3, 5.
pos = pd.Index(b.source_id).get_indexer(members.source_id)  # row of each member in the parent table
# Selection-group index, good and observed flags, and pass groups, as in scripts/boss_selection.py.
s_, good, obs = pd.factorize(b.group)[0], b.good.to_numpy(), b.observed.to_numpy()
pg = selection.boss_pass_groups(b.g, b.mh_xp, b.ra, b.dec)
# Coverage-factor cell: ring x G bin (12-15.5-16.25-16.75-17.1-17.5).
key = b.ring.to_numpy() * 10 + bin_index(b.g, [12.0, 15.5, 16.25, 16.75, 17.1, 17.5])
ones, sens = np.ones(len(b)), {}
for nmin in (2, 3, 5):
    selection.NMIN_GOOD = nmin  # module constant read at call time
    # Final weight (group weight x pass factor), times the ring x G coverage factor, at the members.
    w = selection.group_weights(s_, good, ones) * selection.pass_factor(s_, obs, good, pg, ones)
    w = (w * selection.coverage_factor([key], np.bincount(s_, good)[s_] >= nmin, 1, ones))[pos]
    # Weighted member counts below [Fe/H] = -1.5, below -2, and in total.
    sens[nmin] = [float(w[members.feh.to_numpy() < x].sum()) for x in (-1.5, -2.0)] + [float(w.sum())]
selection.NMIN_GOOD = 3  # restore the adopted value
# Counts relative to those with NMIN_GOOD = 3; then write and print the JSON.
out['coverage_threshold'] = {f'nmin={k}': dict(zip(['<-1.5', '<-2', 'all'], np.round(np.array(v) / np.array(sens[3]), 4).tolist())) for k, v in sens.items()}
json.dump(out, open(ROOT / 'results/selection_gaps.json', 'w'), indent=1)
print(json.dumps(out, indent=1))

#### The figure
(a) good spectra over parent on the sky; (b) XP metallicities of the gaps; (c) non-members.

In [ ]:
# Three equal panels in one row (sizes in inches).
fig, axes = panel_grid(1, 3, width=7, left=.5, right=.1, hgap=.62, bottom=.42, top=.62)
# (a) Per nside-64 pixel (ring order): fraction of the parent with a good spectrum and fraction in uncovered groups.
pix = hp.ang2pix(64, b.ra.to_numpy(), b.dec.to_numpy(), lonlat=True)
per = pd.DataFrame(dict(pix=pix, good=b.good.to_numpy(), unc=~b.covered.to_numpy())).groupby('pix').agg(good=('good', 'mean'), unc=('unc', 'mean'))
# Pixel centres [deg], projected about the LMC centre (orthographic x east, y north [deg]).
ra, dec = hp.pix2ang(64, per.index.to_numpy(), lonlat=True)
px, py = lmc_xy(ra, dec)
sc = axes[0, 0].scatter(px, py, c=np.clip(per.good, 1e-3, 1), s=7.5, marker='s', cmap='viridis', norm=LogNorm(1e-3, 1), lw=0, rasterized=True)
u = per.unc.to_numpy() > 0.5  # pixels with more than half of their parent in uncovered groups: red outlines
axes[0, 0].scatter(px[u], py[u], s=7.5, marker='s', facecolor='none', edgecolor='r', lw=.5)
# East to the left; colour bar above the panel.
axes[0, 0].set(xlim=(12.5, -12.5), ylim=(-12.5, 12.5), xlabel='$x$ (east) [deg]', ylabel='$y$ (north) [deg]')
axes[0, 0].set_title('(a) good spectra / parent (red: uncovered)', pad=22)
fig.colorbar(sc, cax=axes[0, 0].inset_axes([0, 1.03, 1, .04]), orientation='horizontal').ax.xaxis.set_ticks_position('top')
# (b) Fraction below each XP [M/H] threshold: uncovered with its bootstrap band, covered reweighted to the same mix.
ax = axes[0, 1]
ax.fill_between(THRESHOLDS, lo, hi, color=ORANGE, alpha=.25, lw=0)
ax.plot(THRESHOLDS, fu, '-o', ms=2.5, color=ORANGE, label='uncovered groups')
ax.plot(THRESHOLDS, fc, '-', color=BLUE, label='covered, same ring and $G$ mix')
ax.set(yscale='log', xlabel=r'[M/H]$_{\rm XP}$ threshold', ylabel=r'fraction below threshold', xlim=(-2.05, -.45))
ax.set_title('(b) metallicity of the gaps', pad=22)
ax.legend(loc='lower right')
# (c) Weighted per cent of non-members per G bin, one line per ring (offset slightly in x); APOGEE as horizontal lines.
ax = axes[0, 2]
labels = ['$<$15.5', '15.5–16.25', '16.25–16.75', '16.75–17.1', '17.1–17.5']
for r, (name, colour) in enumerate(zip(['0–3°', '3–6°', '6–12°'], ['0.1', BLUE, ORANGE])):
    y = [boss_nm.get((r, k), np.nan) for k in range(5)]
    ax.plot(np.arange(5) + (r - 1) * .08, 100 * np.array(y), '-o', ms=3, color=colour, label=f'BOSS {name}')
for k, (t, s) in enumerate(out['apogee_nonmember'].items()):
    ax.axhline(100 * s['weighted'], color='0.5', ls=['--', ':'][k], lw=.8, label=f'APOGEE {t}')
ax.set(xticks=range(5), ylabel='non-members among good spectra (%)', xlabel='$G$', ylim=(0, 7))
ax.set_xticklabels(labels, rotation=30, fontsize=6)
ax.set_title('(c) non-members', pad=22)
ax.legend(loc='upper left', fontsize=6)
fig.savefig(ROOT / 'figures/paper_selection_gaps.png')
print('Saved paper_selection_gaps.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_selection_gaps.png')

**Figure 2.3.** Coverage of the BOSS parent. (a) Fraction of the parent with a good spectrum in HEALPix
$N_{\rm side}$ = 64 pixels (logarithmic colour scale, 10⁻³ to 1), in the orthographic projection $x$ (east,
increasing to the left), $y$ (north) in degrees; red outlines: pixels with most of their parent in uncovered groups. (b) Fraction of
the parent with [M/H]$_{\rm XP}$ below a threshold, against the threshold (dex; logarithmic $y$ axis), for the uncovered groups
(orange; band: 16th–84th percentiles of a bootstrap over groups) and for covered stars reweighted to the same ring and $G$ mix
(blue). (c) Weighted per cent of good spectra that are not LMC giants (not giants, or $v_{\rm los}$ outside
150–420 km/s) per $G$ bin: BOSS in the rings 0–3° (black), 3–6° (blue) and 6–12° (orange),
and APOGEE in LMC1–17 (dashed grey line) and LMC18–35 (dotted grey line). `scripts/paper_selection_gaps.py`